# Etap III — Parametryzacja Funkcji Impedancji i WTC (Kalibracja Kosztu)

Ten notebook realizuje **Etap III** z `realization_plan.md` dla 3 miast (Dublin, Paryż, Warszawa).

Budujemy macierze czasów podróży (TTM — Travel Time Matrix) przy użyciu R5 oraz transformujemy je do macierzy kosztów uogólnionych (WTC — Whole Travel Chain Cost).

**Wejścia (kanoniczne):** wyłącznie artefakty z Etapu I (`../outputs/etap1/artifacts_index.json`).

**Wyjścia:** `../outputs/etap3/<city>/...` zgodnie z kontraktem z `realization_plan.md`.

Założenia:
- Dobieramy datę wyjazdu dynamicznie z dostępnego zakresu GTFS (preferując środek).
- Macierze TTM budujemy dla próbki jednostek OD (domyślnie k=750 origins × k=750 destinations).
- Fallback: jeśli R5 nie działa, stosujemy walk-only z ograniczeniem czasowym.


In [1]:
from __future__ import annotations

import importlib.util
import json
import os
import sys
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import geopandas as gpd


def _find_repo_root(start: Path) -> Path:
    """Find repository root by walking upwards until `realization_plan.md` is found."""
    p = start.resolve()
    for cand in [p] + list(p.parents):
        if (cand / 'realization_plan.md').exists():
            return cand
    raise FileNotFoundError(f"Cannot find repo root upwards from: {start} (resolved={p})")


try:
    _NOTEBOOK_DIR = Path(__file__).resolve().parent  # type: ignore[name-defined]
except Exception:
    _NOTEBOOK_DIR = Path.cwd()

ROOT = _find_repo_root(_NOTEBOOK_DIR)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.gtfs_utils import (
    choose_departure_datetime_within_gtfs,
    read_gtfs_service_date_range,
    choose_service_day_within_gtfs,
    gtfs_has_any_service_on_date,
)
from scripts.gtfs_merge import merge_gtfs_zips

# r5py (optional, requires Java)
HAS_R5PY = importlib.util.find_spec('r5py') is not None
if HAS_R5PY:
    import r5py

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('geopandas:', gpd.__version__)
print('HAS_R5PY:', HAS_R5PY)
if HAS_R5PY:
    print('r5py:', r5py.__version__)

Python: 3.9.13
pandas: 2.3.3
numpy: 2.0.2
geopandas: 1.0.1
HAS_R5PY: True
r5py: 1.0.0


## 0) Konfiguracja (kontrakt ścieżek) + parametry analizy

Parametry:
- `OD_SCALE_MODE` i `OD_SAMPLE_K` kontrolują próbkowanie jednostek OD.
- `ANALYSIS_DATE_YYYYMMDD` jeśli None, dobieramy automatycznie datę z zakresu GTFS.
- `DEPARTURE_TIME_WINDOW` określa poranne okno wyjazdu (np. 7:30-9:00).


In [2]:
os.chdir(str(ROOT))

ETAP1_DIR = ROOT / 'outputs' / 'etap1'
ARTIFACTS_ETAP1 = ETAP1_DIR / 'artifacts_index.json'
OUT_DIR = ROOT / 'outputs' / 'etap3'
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not ARTIFACTS_ETAP1.exists():
    raise FileNotFoundError(f'Missing Etap1 artifacts index: {ARTIFACTS_ETAP1}')

CITIES = ['dublin', 'paris', 'warsaw']

CITY_CRS_METRIC = {
    'dublin': 'EPSG:2157',
    'paris': 'EPSG:2154',
    'warsaw': 'EPSG:2180',
}

# --- Ścieżki do granic administracyjnych (z Etapu I, cached w Data/) ---
DATA_DIR = ROOT / 'Data'
CITY_ADMIN_BOUNDARY_PATH = {
    'dublin': DATA_DIR / 'Dublin' / 'boundary_admin.geojson',
    'paris': DATA_DIR / 'Paris' / 'boundary_admin.geojson',
    'warsaw': DATA_DIR / 'Warsaw' / 'boundary_admin.geojson',
}

# --- Parametry OD (skalowanie) ---
OD_SCALE_MODE: str = 'sample_k'
OD_SAMPLE_K: int = 750
OD_RANDOM_SEED: int = 42

# Role OD per miasto — 'grid_to_grid' zapewnia porównywalność między miastami
PARIS_OD_ROLE_MODE: str = 'grid_to_grid'

# --- Filtrowanie OD do obszaru obsługi (admin boundary + bliskość przystanków) ---
USE_ADMIN_BOUNDARY_FILTER: bool = True
STOPS_PROXIMITY_BUFFER_M: int = 2000  # komórki dalej niż 2 km od przystanku → odrzucamy

# --- Parametry podróży (R5) ---
ANALYSIS_DATE_YYYYMMDD: Optional[str] = None
DEPARTURE_TIME_WINDOW: Tuple[str, str] = ('07:30:00', '09:00:00')

# Multi-departure (redukcja wrażliwości na headway)
USE_MULTI_DEPARTURES: bool = True
DEPARTURE_TIME_CANDIDATES: Tuple[str, ...] = ('07:30:00', '08:00:00', '08:30:00')
MULTI_DEPARTURE_AGG: str = 'min'

# Walk / transit parametry
WALK_SPEED_MPS: float = 1.25
WALK_SPEED_KMH: float = WALK_SPEED_MPS * 3.6
MAX_WALK_DISTANCE_M: int = 2000  # zwiększone z 1500 dla lepszego pokrycia R5
MAX_TRIP_DURATION_MIN: int = 180

# Parametry fallback (walk-only)
FALLBACK_WALK_ONLY_MAX_TIME_MIN: float = 90.0  # obniżone z 240 — >90 min marszu ≈ nieosiągalne
WALK_DETOUR_FACTOR: float = 1.3  # korekta dystansu euklidesowego → sieciowy (1.2–1.4 typowe w miastach EU)

# QC thresholds
R5_MIN_NONNULL_SHARE: float = 0.01
R5_MIN_NONNULL_ABS: int = 500

# Fill / sparsity — zwiększono MAX_FILL_FRACTION by akceptować fill zamiast pustej macierzy
MAX_FILL_FRACTION: float = 0.95
FILL_MISSING_WITH_WALK: bool = True
DROP_UNSNAPPED_POINTS: bool = False

# R5 departure_time_window: czas (w minutach) wewnętrznej randomizacji R5
# Ustawiony na 10 min, żeby R5 próbkował departures w krótkim oknie wokół
# jawnie podanych godzin (zamiast domyślnych 60 min, które rozszerzały okno do 09:30).
R5_DEPARTURE_TIME_WINDOW_MIN: int = 10

# Snap-to-streets params — obniżone progi for better R5 coverage
SNAP_TO_STREETS_BEFORE_R5: bool = True
SNAP_MAX_DIST_M: float = 2000.0
SNAP_MIN_KEEP_SHARE: float = 0.50

# Diagnostic params
DIAG_DESTINATIONS_FROM_ORIGINS: bool = False
DIAG_SAMPLE_K: int = 200

# --- Mapowanie kolumn atrybutowych per miasto (populacja / zatrudnienie) ---
# Używane do ekstrakcji O_i (popyt) i D_j (podaż) z warstw Etapu I
CITY_POP_COLUMN = {
    'dublin': 'pop',      # z Census 2022 GISCO grid → sjoin do ITM grid (RC3, etap1)
    'paris': 'pop',       # z fra_pd_2020_1km_ASCII_XYZ.csv → cell_id, pop
    'warsaw': 'tot',      # z NSP2021 census → tot = populacja ogółem
}
CITY_JOBS_COLUMN = {
    'dublin': 'employment',   # areal-weighted z Workplace Zones → kolumna 'employment' (Etap I)
    'paris': 'employment',    # areal-weighted z workplace_density (SIRENE) → kolumna 'employment' (Etap I)
    'warsaw': None,           # brak danych o zatrudnieniu na siatce (ograniczenie)
}

print('ROOT:', ROOT)
print('OUT_DIR:', OUT_DIR)
print('OD_SCALE_MODE:', OD_SCALE_MODE, 'k=', OD_SAMPLE_K, 'seed=', OD_RANDOM_SEED)
print('PARIS_OD_ROLE_MODE:', PARIS_OD_ROLE_MODE)
print('USE_ADMIN_BOUNDARY_FILTER:', USE_ADMIN_BOUNDARY_FILTER, 'stop_buffer_m=', STOPS_PROXIMITY_BUFFER_M)
print('R5_DEPARTURE_TIME_WINDOW_MIN:', R5_DEPARTURE_TIME_WINDOW_MIN)
print('HAS_R5PY:', HAS_R5PY)


ROOT: C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters
OUT_DIR: C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\outputs\etap3
OD_SCALE_MODE: sample_k k= 750 seed= 42
PARIS_OD_ROLE_MODE: grid_to_grid
USE_ADMIN_BOUNDARY_FILTER: True stop_buffer_m= 2000
R5_DEPARTURE_TIME_WINDOW_MIN: 10
HAS_R5PY: True


In [3]:
artifacts_etap1 = json.loads(ARTIFACTS_ETAP1.read_text(encoding='utf-8'))


## 1) Funkcje pomocnicze (ładowanie danych, mapowanie ścieżek)


In [4]:
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec='seconds').replace('+00:00', 'Z')


def resolve_artifact_path(p: str) -> Path:
    """Mapuje ścieżkę z artifacts_index.json na istniejący plik lokalny.

    Etap1 może mieć ścieżki absolutne z innego środowiska (np. Dropbox).
    Jeżeli plik nie istnieje, próbujemy podmienić prefiks na lokalny ROOT.
    """
    path = Path(p)
    if path.exists():
        return path

    s = str(p).replace('\\', '/')
    marker = '/outputs/etap1/'
    if marker in s:
        suffix = s.split(marker, 1)[1]
        candidate = ROOT / 'outputs' / 'etap1' / suffix
        if candidate.exists():
            return candidate

    candidate = (ROOT / p).resolve()
    if candidate.exists():
        return candidate

    raise FileNotFoundError(f'Cannot resolve artifact path: {p}')


@dataclass(frozen=True)
class CityInputs:
    city: str
    gtfs_tables: Dict[str, Path]
    spatial_layers: Dict[str, Path]


REQ_GTFS_TABLES = ['stops', 'routes', 'trips', 'stop_times']


def build_inputs(city: str) -> CityInputs:
    gtfs_tables_raw = artifacts_etap1.get('gtfs', {}).get(city, {}).get('tables', {})
    spatial_layers_raw = artifacts_etap1.get('spatial', {}).get(city, {}).get('layers', {})

    if not gtfs_tables_raw:
        raise KeyError(f'No GTFS tables listed for city={city} in artifacts_index.json')

    for t in REQ_GTFS_TABLES:
        if t not in gtfs_tables_raw:
            raise KeyError(f'{city}: missing required GTFS table in Etap1 artifacts: {t}')

    gtfs_tables: Dict[str, Path] = {}
    for k, p in gtfs_tables_raw.items():
        try:
            gtfs_tables[k] = resolve_artifact_path(p)
        except FileNotFoundError:
            if k in REQ_GTFS_TABLES:
                raise

    spatial_layers: Dict[str, Path] = {}
    for layer_name, layer_paths in spatial_layers_raw.items():
        if isinstance(layer_paths, dict) and 'metric_geojson' in layer_paths:
            spatial_layers[layer_name] = resolve_artifact_path(layer_paths['metric_geojson'])

    return CityInputs(city=city, gtfs_tables=gtfs_tables, spatial_layers=spatial_layers)


inputs_by_city: Dict[str, CityInputs] = {c: build_inputs(c) for c in CITIES}
inputs_by_city



def _rel(p) -> str:
    """Konwertuje ścieżkę na względną względem ROOT (Fix 3.6)."""
    try:
        return str(Path(p).relative_to(ROOT))
    except ValueError:
        return str(p)

## 2) Global run_info + inventory (w stylu etap1/etap2)


In [5]:
run_info = {
    'run_utc': utc_now_iso(),
    'root': str(ROOT),
    'out_dir': str(OUT_DIR),
    'inputs': {
        'etap1_artifacts_index': str(ARTIFACTS_ETAP1),
    },
    'analysis': {
        'od_scale_mode': OD_SCALE_MODE,
        'od_sample_k': OD_SAMPLE_K,
        'od_random_seed': OD_RANDOM_SEED,
        'paris_od_role_mode': PARIS_OD_ROLE_MODE,
        'analysis_date_yyyymmdd': ANALYSIS_DATE_YYYYMMDD,
        'departure_time_window': DEPARTURE_TIME_WINDOW,
        'use_admin_boundary_filter': USE_ADMIN_BOUNDARY_FILTER,
        'stops_proximity_buffer_m': STOPS_PROXIMITY_BUFFER_M,
        'walk_speed_mps': WALK_SPEED_MPS,
        'walk_detour_factor': WALK_DETOUR_FACTOR,
        'max_walk_distance_m': MAX_WALK_DISTANCE_M,
        'max_trip_duration_min': MAX_TRIP_DURATION_MIN,
        'fallback_walk_only_max_time_min': FALLBACK_WALK_ONLY_MAX_TIME_MIN,
        'max_fill_fraction': MAX_FILL_FRACTION,
    },
    'versions': {
        'python': sys.version.split()[0],
        'pandas': pd.__version__,
        'numpy': np.__version__,
        'geopandas': gpd.__version__,
        'r5py_available': HAS_R5PY,
    },
    'cities': CITIES,
}

(OUT_DIR / 'run_info.json').write_text(json.dumps(run_info, ensure_ascii=False, indent=2), encoding='utf-8')

inv_rows = []
for city, cin in inputs_by_city.items():
    for name, path in sorted(cin.gtfs_tables.items()):
        inv_rows.append({'city': city, 'type': 'gtfs_table_etap1', 'name': name, 'path': str(path), 'exists': path.exists()})
    for name, path in sorted(cin.spatial_layers.items()):
        inv_rows.append({'city': city, 'type': 'spatial_layer_metric_etap1', 'name': name, 'path': str(path), 'exists': path.exists()})
    # Dodaj boundary do inwentarza
    bp = CITY_ADMIN_BOUNDARY_PATH.get(city)
    if bp:
        inv_rows.append({'city': city, 'type': 'admin_boundary', 'name': 'boundary_admin', 'path': str(bp), 'exists': bp.exists()})

inventory_df = pd.DataFrame(inv_rows)
inventory_df.to_csv(OUT_DIR / 'inventory.csv', index=False)
inventory_df.head(15)

,city,type,name,path,exists
0,dublin,gtfs_table_etap1,agency,outputs\etap1\dublin\gtfs_normalized\agency.csv,True
1,dublin,gtfs_table_etap1,calendar,outputs\etap1\dublin\gtfs_normalized\calendar.csv,True
2,dublin,gtfs_table_etap1,calendar_dates,outputs\etap1\dublin\gtfs_normalized\calendar_...,True
3,dublin,gtfs_table_etap1,feed_info,outputs\etap1\dublin\gtfs_normalized\feed_info...,True
4,dublin,gtfs_table_etap1,routes,outputs\etap1\dublin\gtfs_normalized\routes.csv,True
5,dublin,gtfs_table_etap1,shapes,outputs\etap1\dublin\gtfs_normalized\shapes.csv,True
6,dublin,gtfs_table_etap1,stop_times,outputs\etap1\dublin\gtfs_normalized\stop_time...,True
7,dublin,gtfs_table_etap1,stops,outputs\etap1\dublin\gtfs_normalized\stops.csv,True
8,dublin,gtfs_table_etap1,trips,outputs\etap1\dublin\gtfs_normalized\trips.csv,True
9,dublin,spatial_layer_metric_etap1,grid_1km,outputs\etap1\dublin\spatial\grid_1km_metric.g...,True


## 3) Funkcje walidacji QC (TTM)


In [6]:
def _is_suspicious_constant_times(ttm: pd.DataFrame, round_to: int = 3, top1_share_threshold: float = 0.98, debug: bool = False) -> bool:
    """Heuristic: True if almost all non-null travel times are identical.

    This catches the failure mode where r5 returns a constant capped value (or a single travel time) for most OD pairs.
    """
    if ttm is None or 'travel_time_min' not in ttm.columns:
        return True
    s = pd.to_numeric(ttm['travel_time_min'], errors='coerce').dropna()
    if s.empty:
        return True
    vc = s.round(round_to).value_counts(normalize=True)
    top1_share = float(vc.iloc[0])
    if debug:
        print(f'  QC constant check: top1_share={top1_share:.3f}, threshold={top1_share_threshold}, top_value={vc.index[0]}')
    return top1_share >= float(top1_share_threshold)


def _ensure_crs(gdf: gpd.GeoDataFrame, crs: str) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        raise ValueError('GeoDataFrame has no CRS set')
    if str(gdf.crs) != crs:
        return gdf.to_crs(crs)
    return gdf


def _stable_unit_id_from_centroid_xy(x: float, y: float) -> str:
    # stabilny, czytelny ID (zaokrąglenie do mm w CRS metrycznym)
    return f'cell_{x:.3f}_{y:.3f}'

In [7]:
# --- Filtrowanie jednostek OD do obszaru obsługi transportu zbiorowego ---

def load_admin_boundary(city: str) -> gpd.GeoDataFrame:
    """Wczytaj granicę administracyjną z Etapu I (WGS84 → metryczny CRS miasta)."""
    path = CITY_ADMIN_BOUNDARY_PATH.get(city)
    if not path or not path.exists():
        raise FileNotFoundError(f'Admin boundary not found for {city}: {path}')
    gdf = gpd.read_file(path)
    if gdf.crs is None:
        gdf = gdf.set_crs('EPSG:4326')
    return gdf.to_crs(CITY_CRS_METRIC[city])


def load_gtfs_stops_as_points(city: str) -> gpd.GeoDataFrame:
    """Wczytaj przystanki z Etapu I jako punkty GeoDataFrame w metrycznym CRS."""
    cin = inputs_by_city[city]
    stops_path = cin.gtfs_tables.get('stops')
    if not stops_path or not stops_path.exists():
        raise FileNotFoundError(f'Stops CSV not found for {city}')
    stops = pd.read_csv(stops_path, dtype='string')
    stops['stop_lat'] = pd.to_numeric(stops['stop_lat'], errors='coerce')
    stops['stop_lon'] = pd.to_numeric(stops['stop_lon'], errors='coerce')
    stops = stops.dropna(subset=['stop_lat', 'stop_lon'])
    gdf = gpd.GeoDataFrame(
        stops,
        geometry=gpd.points_from_xy(stops['stop_lon'], stops['stop_lat']),
        crs='EPSG:4326',
    )
    return gdf.to_crs(CITY_CRS_METRIC[city])


def filter_units_to_service_area(
    units_metric: gpd.GeoDataFrame,
    city: str,
    buffer_m: int = STOPS_PROXIMITY_BUFFER_M,
) -> Tuple[gpd.GeoDataFrame, dict]:
    """Filtruj jednostki OD do granicy administracyjnej + bliskości przystanków.

    Krok 1: Zachowaj jednostki, których centroid leży w obrębie granicy administracyjnej.
    Krok 2: Zachowaj jednostki, których centroid jest w zasięgu buffer_m od przystanku GTFS.

    Returns: (filtered_units, qc_dict)
    """
    if units_metric is None or len(units_metric) == 0:
        return units_metric, {'n_input': 0, 'n_after_admin_boundary': 0, 'n_after_stop_proximity': 0}

    n_input = len(units_metric)

    # Krok 1: granica administracyjna
    boundary = load_admin_boundary(city)
    boundary_union = boundary.union_all()

    cents = units_metric.geometry.centroid
    mask_boundary = cents.within(boundary_union)
    units_in_boundary = units_metric.loc[mask_boundary.to_numpy()].copy()
    n_after_boundary = len(units_in_boundary)

    if n_after_boundary == 0:
        print(f'  {city}: WARNING — 0 units within admin boundary! Skipping stop filter.')
        qc = {'n_input': n_input, 'n_after_admin_boundary': 0, 'n_after_stop_proximity': 0, 'kept_share': 0.0}
        return gpd.GeoDataFrame(units_in_boundary, geometry='geometry', crs=units_metric.crs), qc

    # Krok 2: bliskość przystanków (sjoin_nearest z max_distance)
    stops = load_gtfs_stops_as_points(city)

    cents_gdf = gpd.GeoDataFrame(
        {'_orig_idx': units_in_boundary.index},
        geometry=units_in_boundary.geometry.centroid.values,
        crs=units_metric.crs,
    )
    nearest = gpd.sjoin_nearest(
        cents_gdf, stops[['geometry']].copy(), how='left',
        max_distance=float(buffer_m), distance_col='_dist',
    )
    keep_idx = nearest.dropna(subset=['_dist'])['_orig_idx'].unique()
    units_near_stops = units_in_boundary.loc[units_in_boundary.index.isin(keep_idx)].copy()
    n_after_stops = len(units_near_stops)

    qc = {
        'n_input': n_input,
        'n_after_admin_boundary': n_after_boundary,
        'n_after_stop_proximity': n_after_stops,
        'stop_buffer_m': buffer_m,
        'n_stops_loaded': len(stops),
        'kept_share': float(n_after_stops / n_input) if n_input else 0.0,
    }
    print(f'  {city}: service area filter: {n_input} -> {n_after_boundary} (boundary) -> {n_after_stops} (stops <={buffer_m}m)')
    return gpd.GeoDataFrame(units_near_stops, geometry='geometry', crs=units_metric.crs), qc

print('Service area filter functions defined.')

Service area filter functions defined.


## 4) Funkcje ładowania jednostek OD i próbkowania


In [8]:
def load_od_units(city: str, role: str) -> gpd.GeoDataFrame:
    """Ładuje jednostki OD jako geometrię poligonów (z Etapu I).

    role:
    - 'grid' -> grid 1km (Dublin/Warsaw) lub 'pop_grid_1km' dla Paris
    - 'employment' -> employment zones (Paris) / work_zones (Dublin)
    """
    cin = inputs_by_city[city]
    if city == 'dublin':
        if role == 'grid':
            layer_name = 'grid_1km'
        elif role == 'employment':
            layer_name = 'work_zones'
        else:
            raise ValueError(f'dublin: role must be grid or employment, got {role}')
    elif city == 'warsaw':
        if role != 'grid':
            raise ValueError(f'{city}: only role=grid supported')
        layer_name = 'grid_1km'
    elif city == 'paris':
        if role == 'grid':
            layer_name = 'pop_grid_1km'
        elif role == 'employment':
            layer_name = 'employment_zones'
        else:
            raise ValueError('paris: role must be grid or employment')
    else:
        raise ValueError('Unknown city')

    if layer_name not in cin.spatial_layers:
        raise KeyError(f'{city}: missing spatial layer in Etap1 artifacts: {layer_name}')

    gdf = gpd.read_file(cin.spatial_layers[layer_name])
    gdf = _ensure_crs(gdf, CITY_CRS_METRIC[city])
    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notna()].copy()

    # --- Normalizacja atrybutów popytowych → ujednolicone nazwy 'pop' i 'jobs' ---
    pop_col = CITY_POP_COLUMN.get(city)
    jobs_col = CITY_JOBS_COLUMN.get(city)

    if pop_col and pop_col in gdf.columns and 'pop' not in gdf.columns:
        gdf['pop'] = pd.to_numeric(gdf[pop_col], errors='coerce')
    elif pop_col and pop_col in gdf.columns:
        gdf['pop'] = pd.to_numeric(gdf['pop'], errors='coerce')

    if jobs_col and jobs_col in gdf.columns and 'jobs' not in gdf.columns:
        gdf['jobs'] = pd.to_numeric(gdf[jobs_col], errors='coerce')

    # Paris pop_grid: kolumna 'Z' → 'pop' (fallback)
    if city == 'paris' and role == 'grid' and 'pop' not in gdf.columns and 'Z' in gdf.columns:
        gdf['pop'] = pd.to_numeric(gdf['Z'], errors='coerce')

    return gdf


def build_centroids_with_ids(units: gpd.GeoDataFrame, id_col: Optional[str] = None) -> gpd.GeoDataFrame:
    c = units.copy()
    c['geometry'] = c.geometry.centroid
    c['x'] = c.geometry.x
    c['y'] = c.geometry.y

    if id_col and id_col in c.columns:
        c['unit_id'] = c[id_col].astype('string')
    else:
        c['unit_id'] = [
            _stable_unit_id_from_centroid_xy(float(x), float(y))
            for x, y in zip(c['x'].to_numpy(), c['y'].to_numpy())
        ]

    # Zagwarantuj unikalność
    if c['unit_id'].duplicated().any():
        dup = c['unit_id'].duplicated(keep=False)
        c.loc[dup, 'unit_id'] = c.loc[dup, 'unit_id'] + '_' + c.loc[dup].groupby('unit_id').cumcount().astype(str)

    return c[['unit_id', 'x', 'y', 'geometry'] + [col for col in c.columns if col not in {'unit_id','x','y','geometry'}]]


def sample_units(gdf: gpd.GeoDataFrame, k: int, seed: int) -> gpd.GeoDataFrame:
    if len(gdf) <= k:
        return gdf
    return gdf.sample(n=k, random_state=seed).copy()


def write_city_run_info(
    city: str,
    city_dir: Path,
    engine_mode: str,
    notes: Optional[str] = None,
    departure_datetimes: Optional[List[str]] = None,
) -> None:
    info = {
        'run_utc': utc_now_iso(),
        'root': str(ROOT),
        'out_dir': str(OUT_DIR),
        'city': city,
        'analysis': run_info['analysis'],
        'engine_mode': engine_mode,
        'departure_datetimes_used': departure_datetimes,
        'notes': notes,
        'versions': run_info['versions'],
    }
    (city_dir / 'run_info.json').write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding='utf-8')

## 5) Budowa jednostek OD (origins/destinations) per miasto

Dla każdego miasta:
- Dublin/Warsaw: grid → grid
- Paris: grid → employment (lub grid → grid jeśli `PARIS_OD_ROLE_MODE='grid_to_grid'`)

Próbkujemy do k origins × k destinations i zapisujemy jako GeoJSON.


In [9]:
OD_META_ROWS = []
od_by_city = {}

for city in CITIES:
    print(f'\n--- {city.upper()} ---')
    city_dir = OUT_DIR / city
    (city_dir / 'od').mkdir(parents=True, exist_ok=True)
    (city_dir / 'ttm').mkdir(parents=True, exist_ok=True)
    (city_dir / 'wtc').mkdir(parents=True, exist_ok=True)
    (city_dir / 'qc').mkdir(parents=True, exist_ok=True)
    (city_dir / 'reports').mkdir(parents=True, exist_ok=True)

    # --- role modes ---
    if city == 'paris':
        if PARIS_OD_ROLE_MODE == 'grid_to_grid':
            origin_role, dest_role = 'grid', 'grid'
        else:
            origin_role, dest_role = 'grid', 'employment'
    else:
        origin_role, dest_role = 'grid', 'grid'

    units_o = load_od_units(city, origin_role)
    units_d = load_od_units(city, dest_role)

    # Filtruj do obszaru obsługi (admin boundary + bliskość przystanków)
    if USE_ADMIN_BOUNDARY_FILTER:
        units_o, qc_filter_o = filter_units_to_service_area(units_o, city)
        units_d, qc_filter_d = filter_units_to_service_area(units_d, city)
        # Zapisz QC filtrowania
        (city_dir / 'qc' / 'od_filter_origins.json').write_text(
            json.dumps(qc_filter_o, ensure_ascii=False, indent=2), encoding='utf-8')
        (city_dir / 'qc' / 'od_filter_destinations.json').write_text(
            json.dumps(qc_filter_d, ensure_ascii=False, indent=2), encoding='utf-8')

    cent_o = build_centroids_with_ids(units_o)
    cent_d = build_centroids_with_ids(units_d)

    if OD_SCALE_MODE == 'sample_k':
        cent_o_s = sample_units(cent_o, OD_SAMPLE_K, OD_RANDOM_SEED)
        cent_d_s = sample_units(cent_d, OD_SAMPLE_K, OD_RANDOM_SEED)  # fix3.4: symetryczna macierz OD
    else:
        raise NotImplementedError(f'OD_SCALE_MODE not implemented: {OD_SCALE_MODE}')

    origins = cent_o_s[['unit_id', 'geometry']].rename(columns={'unit_id': 'origin_id'}).copy()
    destinations = cent_d_s[['unit_id', 'geometry']].rename(columns={'unit_id': 'dest_id'}).copy()

    origins = gpd.GeoDataFrame(origins, geometry='geometry', crs=CITY_CRS_METRIC[city])
    destinations = gpd.GeoDataFrame(destinations, geometry='geometry', crs=CITY_CRS_METRIC[city])

    # zapis
    origins.to_file(city_dir / 'od' / 'origins.geojson', driver='GeoJSON')
    destinations.to_file(city_dir / 'od' / 'destinations.geojson', driver='GeoJSON')

    # --- Ekstrakcja atrybutów popytowych (O_i / D_j) dla WSZYSTKICH miast ---
    od_units_rows = []

    # Origins: populacja (O_pop)
    if 'pop' in cent_o_s.columns:
        for _, r in cent_o_s.iterrows():
            od_units_rows.append({
                'role': 'origin', 'unit_id': r['unit_id'],
                'O_pop': float(r['pop']) if pd.notna(r.get('pop')) else None,
            })
    else:
        # Zapisz origins bez atrybutu (ograniczenie danych, np. Dublin)
        for _, r in cent_o_s.iterrows():
            od_units_rows.append({
                'role': 'origin', 'unit_id': r['unit_id'],
                'O_pop': None,
            })

    # Destinations: zatrudnienie (D_jobs) lub populacja
    if 'jobs' in cent_d_s.columns:
        for _, r in cent_d_s.iterrows():
            od_units_rows.append({
                'role': 'dest', 'unit_id': r['unit_id'],
                'D_jobs': float(r['jobs']) if pd.notna(r.get('jobs')) else None,
            })
    elif 'pop' in cent_d_s.columns:
        for _, r in cent_d_s.iterrows():
            od_units_rows.append({
                'role': 'dest', 'unit_id': r['unit_id'],
                'D_pop': float(r['pop']) if pd.notna(r.get('pop')) else None,
            })
    else:
        for _, r in cent_d_s.iterrows():
            od_units_rows.append({
                'role': 'dest', 'unit_id': r['unit_id'],
            })

    od_units_df = pd.DataFrame(od_units_rows)
    od_units_df.to_csv(city_dir / 'od' / 'od_units.csv', index=False)

    n_origins_with_pop = od_units_df.loc[od_units_df['role'] == 'origin', 'O_pop'].notna().sum() if 'O_pop' in od_units_df.columns else 0
    n_dest_with_attr = 0
    if 'D_jobs' in od_units_df.columns:
        n_dest_with_attr = od_units_df.loc[od_units_df['role'] == 'dest', 'D_jobs'].notna().sum()
    elif 'D_pop' in od_units_df.columns:
        n_dest_with_attr = od_units_df.loc[od_units_df['role'] == 'dest', 'D_pop'].notna().sum()

    print(f'  od_units.csv: {len(od_units_df)} rows, origins_with_pop={n_origins_with_pop}, dests_with_attr={n_dest_with_attr}')

    # walidacje minimalne
    val = {
        'city': city,
        'crs_metric': CITY_CRS_METRIC[city],
        'origin_role': origin_role,
        'dest_role': dest_role,
        'od_scale_mode': OD_SCALE_MODE,
        'n_origins': int(len(origins)),
        'n_destinations': int(len(destinations)),
        'n_pairs': int(len(origins) * len(destinations)),
        'origins_unique': bool(origins['origin_id'].is_unique),
        'destinations_unique': bool(destinations['dest_id'].is_unique),
        'origins_bounds': list(map(float, origins.total_bounds)),
        'destinations_bounds': list(map(float, destinations.total_bounds)),
        'admin_boundary_filter': USE_ADMIN_BOUNDARY_FILTER,
        'stops_proximity_buffer_m': STOPS_PROXIMITY_BUFFER_M if USE_ADMIN_BOUNDARY_FILTER else None,
        'origins_with_pop': int(n_origins_with_pop),
        'dests_with_attr': int(n_dest_with_attr),
    }
    (city_dir / 'qc' / 'validation_report.json').write_text(
        json.dumps(val, ensure_ascii=False, indent=2), encoding='utf-8')

    OD_META_ROWS.append(val)
    od_by_city[city] = {'origins': origins, 'destinations': destinations,
                        'origin_role': origin_role, 'dest_role': dest_role}

OD_META = pd.DataFrame(OD_META_ROWS)
OD_META


--- DUBLIN ---
  dublin: service area filter: 7087 -> 7087 (boundary) -> 2547 (stops <=2000m)
  dublin: service area filter: 7087 -> 7087 (boundary) -> 2547 (stops <=2000m)
  od_units.csv: 1500 rows, origins_with_pop=0, dests_with_attr=750

--- PARIS ---
  paris: service area filter: 1444 -> 1444 (boundary) -> 1440 (stops <=2000m)
  paris: service area filter: 1444 -> 1444 (boundary) -> 1440 (stops <=2000m)
  od_units.csv: 1500 rows, origins_with_pop=750, dests_with_attr=750

--- WARSAW ---
  warsaw: service area filter: 5040 -> 5040 (boundary) -> 2616 (stops <=2000m)
  warsaw: service area filter: 5040 -> 5040 (boundary) -> 2616 (stops <=2000m)
  od_units.csv: 1500 rows, origins_with_pop=750, dests_with_attr=750


,city,crs_metric,origin_role,dest_role,od_scale_mode,n_origins,n_destinations,n_pairs,origins_unique,destinations_unique,origins_bounds,destinations_bounds,admin_boundary_filter,stops_proximity_buffer_m,origins_with_pop,dests_with_attr
0,dublin,EPSG:2157,grid,grid,sample_k,750,750,562500,True,True,"[662500.0, 661500.0, 733415.5637722069, 790500.0]","[662500.0, 661500.0, 733415.5637722069, 790500.0]",True,2000,0,750
1,paris,EPSG:2154,grid,grid,sample_k,750,750,562500,True,True,"[637863.9981190157, 6841314.962102577, 670909....","[637863.9981190157, 6841314.962102577, 670909....",True,2000,750,750
2,warsaw,EPSG:2180,grid,grid,sample_k,750,750,562500,True,True,"[607615.6848371938, 452811.6081203231, 697420....","[607615.6848371938, 452811.6081203231, 697420....",True,2000,750,750


## 6) Konfiguracja R5 per miasto (GTFS + OSM)

Dla każdego miasta określamy ścieżki do GTFS i OSM (PBF).

Warszawa: jeśli dostępne, mergujemy kilka operatorów (ZTM, KM, WKD, Polregio).


In [10]:
DATA_DIR = ROOT / 'Data'

# Pozwól nadpisać ścieżki OSM/GTFS przez ENV, żeby łatwo uruchomić r5py bez grzebania w kodzie.

def _env_path(var_name: str) -> Optional[Path]:
    v = os.environ.get(var_name)
    if not v:
        return None
    p = Path(v)
    if not p.is_absolute():
        p = (ROOT / p).resolve()
    return p

# gdzie trzymamy zmergowane/spakowane feedy GTFS na potrzeby R5
MERGED_GTFS_DIR = OUT_DIR / '_merged_gtfs'
MERGED_GTFS_DIR.mkdir(parents=True, exist_ok=True)

# Katalog z outputami Etap 1
ETAP1_DIR = ROOT / 'outputs' / 'etap1'

# Surowe sciezki (fallback â€” uzywane jesli Etap 1 niedostepny lub nie uruchomiony)
CITY_R5_INPUTS_RAW: dict = {
    'dublin': {
        'gtfs_zip': DATA_DIR / 'Dublin' / 'GTFS_Realtime.zip',
        'osm_pbf': DATA_DIR / 'Dublin' / 'ireland-and-northern-ireland-latest.osm.pbf',
    },
    'paris': {
        'gtfs_zip': DATA_DIR / 'Paris' / 'gouv_paris_gtfs-export.zip',
        'osm_pbf': DATA_DIR / 'Paris' / 'ile-de-france-260111.osm.pbf',
    },
    'warsaw': {
        # fallback: scalony feed surowy (duzy, bez filtracji do granicy)
        'gtfs_zip': _env_path('GTFS_ZIP_WARSAW') or (DATA_DIR / 'Warsaw' / 'warsaw_merged.zip'),
        'gtfs_zip_alternatives': [
            DATA_DIR / 'Warsaw' / 'warsaw_merged.zip',
            DATA_DIR / 'Warsaw' / 'gtfs - ztm wwa.zip',
            DATA_DIR / 'Warsaw' / 'kolejemazowieckie.zip',
            DATA_DIR / 'Warsaw' / 'wkd.zip',
            DATA_DIR / 'Warsaw' / 'GTFS - polregio.zip',
            DATA_DIR / 'Warsaw' / 'warsaw -mkuran.zip',
        ],
        'osm_pbf': DATA_DIR / 'Warsaw' / 'mazowieckie-260111.osm.pbf',
    },
}


def _etap1_osm_pbf(city: str) -> Optional[Path]:
    """Zwraca sciezke do pliku OSM PBF uzytego przez Etap 1 (mniejszy, dociety do granicy)."""
    rpt_path = ETAP1_DIR / city / 'gtfs_report.json'
    if not rpt_path.exists():
        return None
    try:
        rpt = json.loads(rpt_path.read_text(encoding='utf-8'))
        osm = rpt.get('osm_pbf')
        if osm:
            p = Path(osm)
            if p.exists():
                return p
    except Exception:
        pass
    return None


def _pack_etap1_gtfs_to_zip(city: str) -> Optional[Path]:
    """Pakuje CSV-y z gtfs_normalized Etap 1 do ZIPa gotowego dla r5py.

    Etap 1 zapisal przystanki i relacje juz przefiltrowane do granicy administracyjnej.
    Dodatkowo usuwa tripy z osieroconym service_id (brak w calendar/calendar_dates)
    i kaskadowo czysci stop_times oraz routes â€” naprawia blokujacy problem R5 dla Warszawy.
    Wynik jest cachowany; przebudowywany tylko jesli pliki CSV sa nowsze niz ZIP.
    """
    csv_dir = ETAP1_DIR / city / 'gtfs_normalized'
    if not csv_dir.exists():
        print(f'  _pack_etap1_gtfs_to_zip({city}): brak katalogu {csv_dir}')
        return None

    out_zip = MERGED_GTFS_DIR / f'etap1_{city}_r5ready.gtfs.zip'

    # Cache: pomijamy jesli ZIP jest nowszy niz wszystkie CSV
    if out_zip.exists():
        zip_mtime = out_zip.stat().st_mtime
        csv_paths = list(csv_dir.glob('*.csv'))
        if csv_paths and all(p.stat().st_mtime <= zip_mtime for p in csv_paths):
            print(f'  _pack_etap1_gtfs_to_zip({city}): korzystam z cache {out_zip.name}')
            return out_zip

    # Wczytaj tabele
    tables: dict[str, pd.DataFrame] = {}
    for csv_path in sorted(csv_dir.glob('*.csv')):
        tables[csv_path.stem] = pd.read_csv(csv_path, dtype=str)

    required = {'stops', 'trips', 'stop_times', 'routes'}
    if not required.issubset(tables.keys()):
        print(f'  _pack_etap1_gtfs_to_zip({city}): brak tabel {required - tables.keys()}')
        return None

    # Zbierz wazne service_ids z calendar + calendar_dates
    cal_svcs: set[str] = set()
    if 'calendar' in tables:
        cal_svcs |= set(tables['calendar']['service_id'].dropna().unique())
    if 'calendar_dates' in tables:
        cal_svcs |= set(tables['calendar_dates']['service_id'].dropna().unique())

    # Usun tripy z osieroconym service_id
    trips_before = len(tables['trips'])
    tables['trips'] = tables['trips'][tables['trips']['service_id'].isin(cal_svcs)].copy()
    trips_after = len(tables['trips'])
    if trips_before != trips_after:
        print(f'  Pack GTFS {city}: usunieto {trips_before - trips_after} osieroconych tripow ({trips_before} -> {trips_after})')

    # Kaskada: stop_times -> tylko dla pozostalych tripow
    valid_trip_ids = set(tables['trips']['trip_id'].unique())
    st_before = len(tables['stop_times'])
    tables['stop_times'] = tables['stop_times'][tables['stop_times']['trip_id'].isin(valid_trip_ids)].copy()
    st_after = len(tables['stop_times'])
    if st_before != st_after:
        print(f'  Pack GTFS {city}: usunieto {st_before - st_after} osieroconych stop_times ({st_before} -> {st_after})')

    # Kaskada: routes -> tylko te uzywane przez pozostale tripy
    if 'route_id' in tables['trips'].columns:
        valid_route_ids = set(tables['trips']['route_id'].unique())
        routes_before = len(tables['routes'])
        tables['routes'] = tables['routes'][tables['routes']['route_id'].isin(valid_route_ids)].copy()
        routes_after = len(tables['routes'])
        if routes_before != routes_after:
            print(f'  Pack GTFS {city}: usunieto {routes_before - routes_after} osieroconych routes ({routes_before} -> {routes_after})')

    # Usuń tripy z < 2 stop_times (single-stop trip = niemożliwy transit).
    # R5 rzuca ArrayIndexOutOfBoundsException gdy trip ma 0 lub 1 przystanek.
    _trip_szs = tables['stop_times'].groupby('trip_id').size()
    _valid_min2 = set(_trip_szs[_trip_szs >= 2].index)
    _single_before = len(tables['trips'])
    tables['trips'] = tables['trips'][tables['trips']['trip_id'].isin(_valid_min2)].copy()
    tables['stop_times'] = tables['stop_times'][tables['stop_times']['trip_id'].isin(_valid_min2)].copy()
    _single_after = len(tables['trips'])
    if _single_before != _single_after:
        print(f'  Pack GTFS {city}: usunieto {_single_before - _single_after} tripow z <2 przystankami ({_single_before} -> {_single_after})')

    # Renumeruj stop_sequences do kolejnych 1-based per trip.
    # Naprawia R5 ArrayIndexOutOfBoundsException gdy:
    #   (a) GTFS używa sekwencji 0-based (Paryż: 0..N-1 → max=N-1, R5 alok. int[N-1], próba int[N-1] → AIOBE)
    #   (b) filtracja Etapu 1 zostawiła luki (np. oryginalne [1..14], zostały [1,5,9,14])
    if 'stop_sequence' in tables['stop_times'].columns:
        _st = tables['stop_times'].copy()
        _st['_sq'] = pd.to_numeric(_st['stop_sequence'], errors='coerce')
        _st = _st.sort_values(['trip_id', '_sq']).drop(columns=['_sq'])
        _st['stop_sequence'] = (_st.groupby('trip_id').cumcount() + 1).astype(str)
        tables['stop_times'] = _st.reset_index(drop=True)

    # Zapisz ZIP (tabele -> .txt zgodnie ze specyfikacja GTFS)
    import io as _io
    import zipfile as _zipfile
    with _zipfile.ZipFile(out_zip, 'w', compression=_zipfile.ZIP_DEFLATED) as zf:
        for tname, df in tables.items():
            buf = _io.StringIO()
            df.to_csv(buf, index=False)
            zf.writestr(f'{tname}.txt', buf.getvalue())

    size_mb = out_zip.stat().st_size / (1024 * 1024)
    n_trips = len(tables['trips'])
    n_stops = len(tables['stops'])
    print(f'  Pack GTFS {city}: zapisano {out_zip.name} ({size_mb:.1f} MB), {n_trips} tripow, {n_stops} przystankow')
    return out_zip


# Inputy R5: CITY_R5_INPUTS_RAW jako baza (z fallback-iem dla Warszawy),
# nadpisane outputami Etap 1 (przefiltrowane do granicy, bez orphanow, maly PBF)
CITY_R5_INPUTS: dict = {}
for _c in CITIES:
    _etap1_gtfs = _pack_etap1_gtfs_to_zip(_c)
    _etap1_osm  = _etap1_osm_pbf(_c)
    _raw = dict(CITY_R5_INPUTS_RAW.get(_c, {}))
    _raw['gtfs_zip'] = _etap1_gtfs or _raw.get('gtfs_zip')
    _raw['osm_pbf']  = _etap1_osm  or _raw.get('osm_pbf')
    CITY_R5_INPUTS[_c] = _raw

# FIX (v2026-03f): Dublin.osm.pbf pokrywa tylko hrabstwo Dublin, nie caly obszar
# metropolitalny (Kildare, Meath, Wicklow). Komorki siatki w tych obszarach nie
# moga sie przypinej do sieci drogowej Dublin.osm.pbf i routing R5 zwraca NaN.
# Dla Dublina uzywamy pelnego pliku Ireland PBF (387 MB) jako OSM.
_ireland_pbf = DATA_DIR / 'Dublin' / 'ireland-and-northern-ireland-latest.osm.pbf'
if _ireland_pbf.exists():
    CITY_R5_INPUTS['dublin']['osm_pbf'] = _ireland_pbf
    print(f'  Dublin OSM override -> {_ireland_pbf.name} ({_ireland_pbf.stat().st_size/1e6:.0f} MB)')

# Weryfikacja: pokaz co zostalo zaladowane
for _city, _inp in CITY_R5_INPUTS.items():
    _gtfs = _inp.get('gtfs_zip')
    _osm = _inp.get('osm_pbf')
    _gtfs_nm = Path(_gtfs).name if _gtfs else 'BRAK'
    _osm_nm  = Path(_osm).name  if _osm  else 'BRAK'
    print(f'{_city}: gtfs={_gtfs_nm}, osm={_osm_nm}')


def _build_merged_warsaw_gtfs(candidate_zips: list[Path]) -> Optional[Path]:
    """Merge multiple Warsaw operator feeds into one GTFS zip."""
    existing = [Path(p) for p in candidate_zips if p and Path(p).exists()]
    if len(existing) < 2:
        return None

    out_zip = MERGED_GTFS_DIR / 'warsaw_merged_gtfs.zip'

    # cache: jeśli out jest nowszy niż inputy, nie miksuj ponownie
    if out_zip.exists():
        out_mtime = out_zip.stat().st_mtime
        if all(p.stat().st_mtime <= out_mtime for p in existing):
            return out_zip

    merge_gtfs_zips(existing, out_zip=out_zip)
    return out_zip


def _pick_best_gtfs_zip(candidate_zips: list[Path]) -> Optional[Path]:
    """Wybiera feed GTFS o możliwie późnej dacie max (aktualność), a potem po długości zakresu."""
    best = None
    best_span = None
    best_max = None

    for zp in candidate_zips:
        if not zp or not Path(zp).exists():
            continue
        try:
            rng = read_gtfs_service_date_range(Path(zp))
        except Exception:
            continue
        if rng.is_empty():
            continue

        span = (rng.max_date() - rng.min_date()).days
        maxd = rng.max_date()

        if best is None:
            best, best_span, best_max = Path(zp), span, maxd
            continue

        if (maxd and best_max and maxd > best_max) or (maxd == best_max and span > best_span):
            best, best_span, best_max = Path(zp), span, maxd

    return best


def _departure_datetimes_for_city(city: str, gtfs_zip: Path) -> list[datetime]:
    """Pick one or multiple departure datetimes on an actual GTFS service day."""
    if ANALYSIS_DATE_YYYYMMDD:
        base = ANALYSIS_DATE_YYYYMMDD
    else:
        # Use 'min' (mindle of GTFS range) for robustness — avoids boundary conditions
        # where R5 considers edge dates as "outside time range" due to feed_info.txt constraints.
        base = choose_service_day_within_gtfs(gtfs_zip, prefer_weekdays=True, prefer='min')

    d = datetime.strptime(base, '%Y%m%d').date()

    if USE_MULTI_DEPARTURES:
        out = []
        for hhmmss in DEPARTURE_TIME_CANDIDATES:
            hh, mm, ss = (int(x) for x in hhmmss.split(':'))
            out.append(datetime(d.year, d.month, d.day, hh, mm, ss))
        return out

    hhmmss = DEPARTURE_TIME_WINDOW[0]
    hh, mm, ss = (int(x) for x in hhmmss.split(':'))
    return [datetime(d.year, d.month, d.day, hh, mm, ss)]


def _aggregate_multi_departure(ttms: list[pd.DataFrame], how: str = 'min') -> pd.DataFrame:
    """Aggregate multiple ttm dataframes into one by OD pair."""
    if not ttms:
        return pd.DataFrame(columns=['origin_id', 'dest_id', 'travel_time_min'])

    base = pd.concat(ttms, ignore_index=True)
    base['travel_time_min'] = pd.to_numeric(base['travel_time_min'], errors='coerce')

    if how == 'median':
        return base.groupby(['origin_id', 'dest_id'], as_index=False).agg(travel_time_min=('travel_time_min', 'median'))

    return base.groupby(['origin_id', 'dest_id'], as_index=False).agg(travel_time_min=('travel_time_min', 'min'))


# Filtr snap R5: weryfikacja, które OD punkty R5 potrafi snap-ować do sieci

def _r5_filter_snappable_points(
    tn,
    pts_wgs84: gpd.GeoDataFrame,
    id_col: str = 'id',
    departure: Optional[datetime] = None,
    require_as_origin_and_dest: bool = True,
) -> tuple[gpd.GeoDataFrame, dict]:
    """Filter points to those that r5 can snap to the street network.

    Works without pyrosm and avoids CRS mismatch issues because we pass WGS84.

    Returns: (pts_kept_wgs84, qc)
    """
    if pts_wgs84 is None or len(pts_wgs84) == 0:
        return pts_wgs84, {'enabled': True, 'reason': 'empty_input', 'n_before': 0, 'n_after': 0}

    if departure is None:
        departure = datetime(2025, 1, 1, 8, 0, 0)

    # small self-matrix (WALK network only)
    try:
        ttm_snap = r5py.TravelTimeMatrix(
            transport_network=tn,
            origins=pts_wgs84[[id_col, 'geometry']].copy(),
            destinations=pts_wgs84[[id_col, 'geometry']].copy(),
            departure=departure,
            departure_time_window=timedelta(minutes=1),
            transport_modes=[r5py.TransportMode.WALK],
            max_time=timedelta(minutes=15),  # zwiększone z 5 min — więcej czasu na snap
            speed_walking=WALK_SPEED_KMH,
            snap_to_network=True,
        )
        if not isinstance(ttm_snap, pd.DataFrame):
            ttm_snap = pd.DataFrame(ttm_snap)
    except Exception as e:
        return pts_wgs84, {'enabled': False, 'reason': f'r5_snap_probe_failed:{e}'}

    if 'from_id' not in ttm_snap.columns or 'to_id' not in ttm_snap.columns:
        return pts_wgs84, {'enabled': False, 'reason': 'r5_snap_probe_missing_columns'}

    from_ids = set(ttm_snap['from_id'].astype('string').unique())
    to_ids = set(ttm_snap['to_id'].astype('string').unique())

    if require_as_origin_and_dest:
        keep = from_ids & to_ids
    else:
        keep = from_ids | to_ids

    before = int(len(pts_wgs84))
    kept = gpd.GeoDataFrame(
        pts_wgs84[pts_wgs84[id_col].astype('string').isin(keep)].copy(),
        geometry='geometry',
        crs=pts_wgs84.crs,
    )
    after = int(len(kept))

    qc = {
        'enabled': True,
        'n_before': before,
        'n_after': after,
        'dropped': before - after,
        'kept_share': float(after / before) if before else 0.0,
        'require_as_origin_and_dest': bool(require_as_origin_and_dest),
    }
    return kept, qc

  _pack_etap1_gtfs_to_zip(dublin): korzystam z cache etap1_dublin_r5ready.gtfs.zip
  _pack_etap1_gtfs_to_zip(paris): korzystam z cache etap1_paris_r5ready.gtfs.zip
  _pack_etap1_gtfs_to_zip(warsaw): korzystam z cache etap1_warsaw_r5ready.gtfs.zip
  Dublin OSM override -> ireland-and-northern-ireland-latest.osm.pbf (387 MB)
dublin: gtfs=etap1_dublin_r5ready.gtfs.zip, osm=ireland-and-northern-ireland-latest.osm.pbf
paris: gtfs=etap1_paris_r5ready.gtfs.zip, osm=ile-de-france-260111.osm.pbf
warsaw: gtfs=etap1_warsaw_r5ready.gtfs.zip, osm=mazowieckie-260111.osm.pbf


## 7) Fallback walk-only (jeśli R5 nie działa)

W razie problemów z R5, obliczamy macierz walk-only (dystans euklidesowy / prędkość pieszego).


In [11]:
def iter_fallback_walk_only_batches(
    origins: gpd.GeoDataFrame,
    destinations: gpd.GeoDataFrame,
    batch_size_origins: int = 60,
):
    """Yield walk-only TTM batches to keep RAM low.

    Each yielded chunk is a DataFrame with columns: origin_id, dest_id, travel_time_min.
    Uses float32 internally to reduce memory pressure.
    Applies WALK_DETOUR_FACTOR to correct euclidean distance to network distance.
    """
    o = origins[['origin_id', 'geometry']].copy()
    d = destinations[['dest_id', 'geometry']].copy()

    d_xy = np.vstack([d.geometry.x.to_numpy(), d.geometry.y.to_numpy()]).T.astype('float32')
    dest_ids = d['dest_id'].to_numpy()
    cap_min = float(FALLBACK_WALK_ONLY_MAX_TIME_MIN)
    detour = float(WALK_DETOUR_FACTOR)

    for start in range(0, len(o), int(batch_size_origins)):
        oo = o.iloc[start:start + int(batch_size_origins)].copy()
        o_xy = np.vstack([oo.geometry.x.to_numpy(), oo.geometry.y.to_numpy()]).T.astype('float32')

        dx = o_xy[:, None, 0] - d_xy[None, :, 0]
        dy = o_xy[:, None, 1] - d_xy[None, :, 1]
        dist_m = np.sqrt(dx * dx + dy * dy, dtype=np.float32) * detour

        t_min = dist_m / float(WALK_SPEED_MPS) / 60.0
        t_min = np.clip(t_min, 0.0, cap_min).astype('float32')

        origin_ids = oo['origin_id'].to_numpy()
        yield pd.DataFrame(
            {
                'origin_id': np.repeat(origin_ids, len(dest_ids)),
                'dest_id': np.tile(dest_ids, len(origin_ids)),
                'travel_time_min': t_min.reshape(-1),
            }
        )


def write_parquet_in_batches(parquet_path: Path, batches) -> None:
    """Write parquet from a generator/iterable of DataFrames without concatenating into RAM."""
    parquet_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        import pyarrow as pa
        import pyarrow.parquet as pq
    except Exception:
        dfs = list(batches)
        if not dfs:
            pd.DataFrame(columns=['origin_id', 'dest_id', 'travel_time_min']).to_parquet(parquet_path, index=False)
            return
        pd.concat(dfs, ignore_index=True).to_parquet(parquet_path, index=False)
        return

    writer = None
    try:
        wrote_any = False
        for b in batches:
            if b is None or len(b) == 0:
                continue
            table = pa.Table.from_pandas(b, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(parquet_path, table.schema, compression='snappy')
            writer.write_table(table)
            wrote_any = True

        if not wrote_any:
            empty = pa.Table.from_pydict({'origin_id': [], 'dest_id': [], 'travel_time_min': []})
            pq.write_table(empty, parquet_path, compression='snappy')
    finally:
        if writer is not None:
            writer.close()

def _walk_time_minutes_for_pairs(
    origins: gpd.GeoDataFrame,
    destinations: gpd.GeoDataFrame,
    origin_ids: pd.Series,
    dest_ids: pd.Series,
) -> np.ndarray:
    """Compute walk times [min] for given OD id pairs in metric CRS.

    Applies WALK_DETOUR_FACTOR to approximate network distance from euclidean.
    Returns NaN for pairs with unmatched IDs (instead of silently mapping to index 0).
    """
    o = origins[['origin_id', 'geometry']].copy()
    d = destinations[['dest_id', 'geometry']].copy()

    o_xy = np.vstack([o.geometry.x.to_numpy(), o.geometry.y.to_numpy()]).T.astype('float32')
    d_xy = np.vstack([d.geometry.x.to_numpy(), d.geometry.y.to_numpy()]).T.astype('float32')

    o_index = {str(i): idx for idx, i in enumerate(o['origin_id'].astype('string').to_numpy())}
    d_index = {str(i): idx for idx, i in enumerate(d['dest_id'].astype('string').to_numpy())}

    oi = origin_ids.astype('string').map(lambda x: o_index.get(str(x), -1)).to_numpy()
    di = dest_ids.astype('string').map(lambda x: d_index.get(str(x), -1)).to_numpy()

    # Zamiast cichej podmiany -1→0, logujemy i ustawiamy NaN
    bad_mask = (oi < 0) | (di < 0)
    if bad_mask.any():
        n_bad = int(bad_mask.sum())
        print(f'  WARNING: _walk_time_minutes_for_pairs: {n_bad} pairs with unmatched IDs -> NaN')
        oi = np.where(oi < 0, 0, oi)
        di = np.where(di < 0, 0, di)

    detour = float(WALK_DETOUR_FACTOR)
    dx = o_xy[oi, 0] - d_xy[di, 0]
    dy = o_xy[oi, 1] - d_xy[di, 1]
    dist_m = np.sqrt(dx * dx + dy * dy, dtype=np.float32) * detour

    t_min = dist_m / float(WALK_SPEED_MPS) / 60.0
    t_min = np.where(t_min <= float(FALLBACK_WALK_ONLY_MAX_TIME_MIN), t_min, np.nan).astype('float32')

    # Ustaw NaN dla par z nieznalezionymi ID
    if bad_mask.any():
        t_min[bad_mask] = np.nan

    return t_min


def _fill_missing_pairs_with_walk(
    ttm: pd.DataFrame,
    origins: gpd.GeoDataFrame,
    destinations: gpd.GeoDataFrame,
) -> pd.DataFrame:
    """Complete the OD matrix by filling missing travel_time_min with walk-only times.

    - Keeps any available R5 `travel_time_min` values.
    - Fills only missing pairs with network-corrected walk time.
    - Assumes `origins`/`destinations` are metric CRS.

    Returns a full |O|x|D| DataFrame with columns: origin_id, dest_id, travel_time_min.
    """
    o_ids = origins['origin_id'].astype('string').to_numpy()
    d_ids = destinations['dest_id'].astype('string').to_numpy()
    full = pd.MultiIndex.from_product([o_ids, d_ids], names=['origin_id', 'dest_id']).to_frame(index=False)

    if ttm is None or len(ttm) == 0:
        merged = full
        merged['travel_time_min'] = np.nan
    else:
        base = ttm[['origin_id', 'dest_id', 'travel_time_min']].copy()
        base['origin_id'] = base['origin_id'].astype('string')
        base['dest_id'] = base['dest_id'].astype('string')
        merged = full.merge(base, on=['origin_id', 'dest_id'], how='left')

    merged['is_walk_fill'] = False  # RC5: flaga par uzupełnionych walk-fallback
    miss = merged['travel_time_min'].isna()
    if not bool(miss.any()):
        return merged

    merged.loc[miss, 'travel_time_min'] = _walk_time_minutes_for_pairs(
        origins=origins,
        destinations=destinations,
        origin_ids=merged.loc[miss, 'origin_id'],
        dest_ids=merged.loc[miss, 'dest_id'],
    )
    merged.loc[miss, 'is_walk_fill'] = True  # RC5
    return merged

## 8) Obliczanie macierzy TTM per miasto

Dla każdego miasta:
1. Próbujemy R5 (jeśli dostępny).
2. W razie błędu lub złych danych → fallback walk-only.
3. Zapisujemy TTM jako parquet + QC summary.

**UWAGA:** Ten krok wymaga Java (dla R5) i może być czasochwonny (szczególnie dla dużych OD).


In [12]:
# First, let's check GTFS date ranges for all cities
print('=== GTFS Date Ranges & Departure Selection ===')
for city in CITIES:
    r5_inputs = CITY_R5_INPUTS[city]
    gtfs_zip = r5_inputs.get('gtfs_zip')

    if city == 'warsaw' and (not gtfs_zip or not Path(gtfs_zip).exists()):
        alts = r5_inputs.get('gtfs_zip_alternatives', [])
        existing_alts = [Path(p) for p in alts if p and Path(p).exists()]
        if len(existing_alts) >= 2:
            merged = _build_merged_warsaw_gtfs(existing_alts)
            if merged:
                gtfs_zip = merged
        if not gtfs_zip or not Path(gtfs_zip).exists():
            gtfs_zip = _pick_best_gtfs_zip(existing_alts) if existing_alts else None

    if gtfs_zip and Path(gtfs_zip).exists():
        try:
            rng = read_gtfs_service_date_range(Path(gtfs_zip))
            # Also compute what departure day will be chosen
            chosen_day = choose_service_day_within_gtfs(gtfs_zip, prefer_weekdays=True, prefer='min')
            chosen_date = datetime.strptime(chosen_day, '%Y%m%d').date()
            in_range = (rng.min_date() <= chosen_date <= rng.max_date()) if not rng.is_empty() else False
            status = '✅' if in_range else '❌ OUT-OF-RANGE'
            print(f'{city.upper():10s}: GTFS range {rng.min_yyyymmdd} to {rng.max_yyyymmdd} | chosen departure day: {chosen_day} {status}')
            print(f'{"":10s}  has_calendar={rng.has_calendar}, has_calendar_dates={rng.has_calendar_dates}')
            # Check service on chosen day
            has_svc = gtfs_has_any_service_on_date(gtfs_zip, chosen_day)
            print(f'{"":10s}  service active on {chosen_day}: {has_svc}')
            if not has_svc:
                print(f'  ⚠️ WARNING: No GTFS service on chosen day {chosen_day}! R5 will route walk-only.')
        except Exception as e:
            print(f'{city.upper():10s}: Error reading dates - {e}')
    else:
        print(f'{city.upper():10s}: GTFS not found')

print()


=== GTFS Date Ranges & Departure Selection ===
DUBLIN    : GTFS range 20260109 to 20261212 | chosen departure day: 20260109 ✅
            has_calendar=True, has_calendar_dates=True
            service active on 20260109: True
PARIS     : GTFS range 20260127 to 20260228 | chosen departure day: 20260127 ✅
            has_calendar=True, has_calendar_dates=True
            service active on 20260127: True
WARSAW    : GTFS range 20251113 to 20251122 | chosen departure day: 20251113 ✅
            has_calendar=True, has_calendar_dates=True
            service active on 20251113: True



In [13]:
TTM_QC_ROWS = []

for city in CITIES:
    print(f'\n=== {city.upper()} ===')
    city_dir = OUT_DIR / city
    od_data = od_by_city[city]
    origins = od_data['origins']
    destinations = od_data['destinations']

    # NEW: diagnostic mode: destinations come from origins (O->O)
    if DIAG_DESTINATIONS_FROM_ORIGINS:
        diag_o = origins.copy()
        if len(diag_o) > int(DIAG_SAMPLE_K):
            diag_o = diag_o.sample(n=int(DIAG_SAMPLE_K), random_state=OD_RANDOM_SEED).copy()
        destinations = diag_o.rename(columns={'origin_id': 'dest_id'}).copy()
        destinations = gpd.GeoDataFrame(destinations, geometry='geometry', crs=origins.crs)
        print(f'DIAG_DESTINATIONS_FROM_ORIGINS=True -> using destinations from origins sample: |O|={len(diag_o)} |D|={len(destinations)}')

    # Keep original OD sets for filling/statistics
    origins_full = origins.copy()
    destinations_full = destinations.copy()

    engine_mode = 'walk_only_fallback'  # default
    departures_used: list = []  # track for reproducibility
    warnings = []
    ttm = None

    # Try R5 if available
    if HAS_R5PY:
        r5_inputs = CITY_R5_INPUTS[city]
        gtfs_zip = r5_inputs.get('gtfs_zip')
        osm_pbf = r5_inputs.get('osm_pbf')

        # Warsaw special case: if primary gtfs_zip doesn't exist, merge or pick best from alternatives
        if city == 'warsaw' and (not gtfs_zip or not Path(gtfs_zip).exists()):
            alts = r5_inputs.get('gtfs_zip_alternatives', [])
            existing_alts = [Path(p) for p in alts if p and Path(p).exists()]
            print(f'Warsaw primary GTFS not found ({gtfs_zip}), checking {len(existing_alts)} alternatives...')
            if len(existing_alts) >= 2:
                merged = _build_merged_warsaw_gtfs(existing_alts)
                if merged:
                    gtfs_zip = merged
                    print(f'Using auto-merged GTFS: {gtfs_zip}')
            if not gtfs_zip or not Path(gtfs_zip).exists():
                gtfs_zip = _pick_best_gtfs_zip(existing_alts)
                if gtfs_zip:
                    print(f'Using best single GTFS: {gtfs_zip}')
        elif city == 'warsaw' and gtfs_zip and Path(gtfs_zip).exists():
            print(f'Using primary GTFS: {gtfs_zip}')

        if gtfs_zip and gtfs_zip.exists() and osm_pbf and osm_pbf.exists():
            try:
                # Service-day-aware departure selection (optionally multi-departure)
                departures = _departure_datetimes_for_city(city, gtfs_zip)
                departures_used = [str(d) for d in departures]
                print(f'Departures: {departures} (service-day-aware)')

                # Build R5 transport network
                print(f'Building R5 transport network for {city}...')
                tn = r5py.TransportNetwork(osm_pbf=osm_pbf, gtfs=[gtfs_zip])

                # Prepare origins/destinations for r5py (must be in WGS84)
                o_wgs84_full = gpd.GeoDataFrame(
                    origins.to_crs('EPSG:4326')[['origin_id', 'geometry']].rename(columns={'origin_id': 'id'}),
                    geometry='geometry',
                    crs='EPSG:4326',
                )
                d_wgs84_full = gpd.GeoDataFrame(
                    destinations.to_crs('EPSG:4326')[['dest_id', 'geometry']].rename(columns={'dest_id': 'id'}),
                    geometry='geometry',
                    crs='EPSG:4326',
                )

                # Hard filter to truly snappable street points using r5 itself
                if SNAP_TO_STREETS_BEFORE_R5:
                    probe_departure = departures[0] if departures else datetime(2025, 1, 1, 8, 0, 0)

                    o_wgs84, qc_o = _r5_filter_snappable_points(tn, o_wgs84_full, id_col='id', departure=probe_departure)
                    d_wgs84, qc_d = _r5_filter_snappable_points(tn, d_wgs84_full, id_col='id', departure=probe_departure)

                    o_wgs84 = gpd.GeoDataFrame(o_wgs84, geometry='geometry', crs='EPSG:4326')
                    d_wgs84 = gpd.GeoDataFrame(d_wgs84, geometry='geometry', crs='EPSG:4326')

                    (city_dir / 'qc' / 'r5_snap_filter_origins.json').write_text(json.dumps(qc_o, indent=2), encoding='utf-8')
                    (city_dir / 'qc' / 'r5_snap_filter_destinations.json').write_text(json.dumps(qc_d, indent=2), encoding='utf-8')

                    if qc_o.get('enabled') and qc_o.get('kept_share', 1.0) < float(SNAP_MIN_KEEP_SHARE):
                        warnings.append(
                            f"R5 snap-filter would drop too many origins (kept_share={qc_o.get('kept_share'):.2f}); proceeding without filtering"
                        )
                        o_wgs84 = o_wgs84_full
                    if qc_d.get('enabled') and qc_d.get('kept_share', 1.0) < float(SNAP_MIN_KEEP_SHARE):
                        warnings.append(
                            f"R5 snap-filter would drop too many destinations (kept_share={qc_d.get('kept_share'):.2f}); proceeding without filtering"
                        )
                        d_wgs84 = d_wgs84_full

                    try:
                        gpd.GeoDataFrame(o_wgs84, geometry='geometry', crs='EPSG:4326').to_file(
                            city_dir / 'od' / 'origins_r5_snapped.geojson', driver='GeoJSON'
                        )
                        gpd.GeoDataFrame(d_wgs84, geometry='geometry', crs='EPSG:4326').to_file(
                            city_dir / 'od' / 'destinations_r5_snapped.geojson', driver='GeoJSON'
                        )
                    except Exception:
                        pass
                else:
                    o_wgs84, d_wgs84 = o_wgs84_full, d_wgs84_full

                # Compute one or multiple TTMs and aggregate
                # max_time_walking: spójne z MAX_WALK_DISTANCE_M (bez sztucznego min 45 min)
                _max_walk_time_min = int(MAX_WALK_DISTANCE_M / WALK_SPEED_MPS / 60) + 5
                ttms = []
                for departure in departures:
                    print(f'Computing travel times for {len(o_wgs84)} origins x {len(d_wgs84)} destinations at {departure}...')
                    ttm_raw = r5py.TravelTimeMatrix(
                        transport_network=tn,
                        origins=o_wgs84,
                        destinations=d_wgs84,
                        departure=departure,
                        departure_time_window=timedelta(minutes=R5_DEPARTURE_TIME_WINDOW_MIN),
                        transport_modes=[r5py.TransportMode.TRANSIT],
                        access_modes=[r5py.TransportMode.WALK],
                        egress_modes=[r5py.TransportMode.WALK],
                        max_time=timedelta(minutes=MAX_TRIP_DURATION_MIN),
                        max_time_walking=timedelta(minutes=_max_walk_time_min),
                        speed_walking=WALK_SPEED_KMH,
                        snap_to_network=True,
                    )

                    if not isinstance(ttm_raw, pd.DataFrame):
                        ttm_raw = pd.DataFrame(ttm_raw)

                    ttmp = ttm_raw.rename(columns={'from_id': 'origin_id', 'to_id': 'dest_id', 'travel_time': 'travel_time_min'})
                    ttmp['travel_time_min'] = pd.to_numeric(ttmp['travel_time_min'], errors='coerce')
                    ttms.append(ttmp[['origin_id','dest_id','travel_time_min']])

                if len(ttms) > 1:
                    ttm = _aggregate_multi_departure(ttms, how=MULTI_DEPARTURE_AGG)
                    print(f'Aggregated multi-departure TTM rows: {len(ttm)}')
                else:
                    ttm = ttms[0]

                print(f'After conversion: {len(ttm)} rows, non-null travel_time: {pd.to_numeric(ttm["travel_time_min"], errors="coerce").notna().sum()}')

                # QC checks
                is_suspicious = _is_suspicious_constant_times(ttm, debug=True)
                nonnull_cnt = int(pd.to_numeric(ttm['travel_time_min'], errors='coerce').notna().sum())
                nonnull_share = float(pd.to_numeric(ttm['travel_time_min'], errors='coerce').notna().mean())

                is_sparse = (nonnull_share < float(R5_MIN_NONNULL_SHARE)) and (nonnull_cnt < int(R5_MIN_NONNULL_ABS))
                print(f'  QC sparse check: nonnull_share={nonnull_share:.3f}, threshold={R5_MIN_NONNULL_SHARE}; nonnull_cnt={nonnull_cnt}, min_abs={R5_MIN_NONNULL_ABS}')
                print(f'QC checks: suspicious_constant={is_suspicious}, too_sparse={is_sparse}')

                if is_suspicious:
                    warnings.append('R5 returned suspicious constant times')
                    print('  -> Rejecting: suspicious constant times')
                    ttm = None
                elif is_sparse:
                    warnings.append('R5 TTM extremely sparse (will fallback)')
                    print('  -> Rejecting: extremely sparse')
                    ttm = None
                else:
                    engine_mode = 'r5py'
                    print(f'R5 TTM accepted: {len(ttm)} rows (nonnull={nonnull_cnt}, share={nonnull_share:.4f})')

                    # POST-HOC TRANSIT VALIDATION
                    s_check = pd.to_numeric(ttm['travel_time_min'], errors='coerce').dropna()
                    median_tt = float(s_check.median()) if len(s_check) else 0.0
                    print(f'  Transit sanity: median_travel_time={median_tt:.1f} min, nonnull={len(s_check)}')

                    try:
                        _rng_check = read_gtfs_service_date_range(gtfs_zip)
                        if not _rng_check.is_empty():
                            _dep_date = departures[0].date() if departures else None
                            _min_d = _rng_check.min_date()
                            _max_d = _rng_check.max_date()
                            if _dep_date and (_dep_date < _min_d or _dep_date > _max_d):
                                warnings.append(
                                    f'CRITICAL: Departure date {_dep_date} is outside GTFS range '
                                    f'[{_min_d}, {_max_d}]. Results are likely WALK-ONLY, not transit!'
                                )
                                print(f'  CRITICAL: departure {_dep_date} outside GTFS [{_min_d},{_max_d}]!')
                    except Exception:
                        pass

                    if DROP_UNSNAPPED_POINTS:
                        kept_o = set(ttm['origin_id'].astype('string').unique())
                        kept_d = set(ttm['dest_id'].astype('string').unique())
                        n_o0, n_d0 = len(origins), len(destinations)
                        origins = origins[origins['origin_id'].astype('string').isin(kept_o)].copy()
                        destinations = destinations[destinations['dest_id'].astype('string').isin(kept_d)].copy()
                        if len(origins) != n_o0 or len(destinations) != n_d0:
                            warnings.append(f'Dropped unsnapped OD points: origins {n_o0}->{len(origins)}, dest {n_d0}->{len(destinations)}')
                            print(f'Dropped unsnapped points: origins {n_o0}->{len(origins)}, dest {n_d0}->{len(destinations)}')

                    # Complete matrix with walk fallback (keeps R5 values where available)
                    if FILL_MISSING_WITH_WALK:
                        fill_origins = origins_full
                        fill_destinations = destinations_full

                        expected_pairs = int(len(fill_origins) * len(fill_destinations))
                        computed_pairs = int(len(ttm))

                        print(
                            f'Fill plan: |O|={len(fill_origins)}, |D|={len(fill_destinations)}, '
                            f'expected_pairs={expected_pairs}, r5_rows={computed_pairs}, r5_nonnull={nonnull_cnt}'
                        )

                        ttm_filled = _fill_missing_pairs_with_walk(ttm, fill_origins, fill_destinations)

                        sfill = pd.to_numeric(ttm_filled['travel_time_min'], errors='coerce')
                        filled_nonnull = int(sfill.notna().sum())
                        filled_by_walk = int(filled_nonnull - nonnull_cnt)
                        fill_fraction = float(filled_by_walk / filled_nonnull) if filled_nonnull else 1.0
                        print(f'Fill stats: filled_nonnull={filled_nonnull}, filled_by_walk~={filled_by_walk}, fill_fraction~={fill_fraction:.3f}')

                        if fill_fraction > float(MAX_FILL_FRACTION):
                            warnings.append(f'Fill fraction {fill_fraction:.3f} > {MAX_FILL_FRACTION}; keeping sparse R5 matrix (no fill) to avoid walk-only dominance')
                            print(f'Fill fraction too high -> NOT filling to avoid walk-only dominance')
                            engine_mode = 'r5py_sparse'
                        else:
                            ttm = ttm_filled
                            engine_mode = 'r5py_plus_walk_fill'
                            print(f'Filled matrix size: {len(ttm)} rows')

            except Exception as e:
                warnings.append(f'R5 failed: {e}')
                print(f'R5 failed: {e}')
                ttm = None

    # Fallback to walk-only if R5 not available or failed
    if ttm is None:
        print(f'Using walk-only fallback for {city}')
        warnings.append('Using walk-only fallback (R5 not available or failed)')
        engine_mode = 'walk_only_fallback'

        batches = iter_fallback_walk_only_batches(origins, destinations)
        ttm_path = city_dir / 'ttm' / 'ttm_raw.parquet'
        write_parquet_in_batches(ttm_path, batches)
        ttm = pd.read_parquet(ttm_path)

    # Write TTM (always overwrite to ensure latest results)
    if ttm is not None:
        ttm.to_parquet(city_dir / 'ttm' / 'ttm_raw.parquet', index=False)

    # QC summary
    s = pd.to_numeric(ttm['travel_time_min'], errors='coerce')
    qc_row = {
        'city': city,
        'engine_mode': engine_mode,
        'n_rows': int(len(ttm)),
        'travel_time_min_nonnull_share': float(s.notna().mean()),
        'travel_time_min_min': float(s.min()) if s.notna().any() else None,
        'travel_time_min_median': float(s.median()) if s.notna().any() else None,
        'travel_time_min_p90': float(s.quantile(0.9)) if s.notna().any() else None,
        'travel_time_min_max': float(s.max()) if s.notna().any() else None,
        'share_capped_or_max_time': float((s >= MAX_TRIP_DURATION_MIN).mean()) if s.notna().any() else 0.0,
    }
    TTM_QC_ROWS.append(qc_row)

    pd.DataFrame([qc_row]).to_csv(city_dir / 'ttm' / 'ttm_qc_summary.csv', index=False)

    if warnings:
        (city_dir / 'qc' / 'warnings.md').write_text('\n'.join(f'- {w}' for w in warnings) + '\n', encoding='utf-8')
    else:
        if (city_dir / 'qc' / 'warnings.md').exists():
            (city_dir / 'qc' / 'warnings.md').unlink()

    write_city_run_info(city, city_dir, engine_mode, departure_datetimes=departures_used or None)
    print(f'Finished {city}: engine_mode={engine_mode}')

TTM_QC = pd.DataFrame(TTM_QC_ROWS)
TTM_QC


=== DUBLIN ===
Departures: [datetime.datetime(2026, 1, 9, 7, 30), datetime.datetime(2026, 1, 9, 8, 0), datetime.datetime(2026, 1, 9, 8, 30)] (service-day-aware)
Building R5 transport network for dublin...


C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\.venv\lib\site-packages\r5py\r5\regional_task.py:266: RuntimeWarning: The provided departure time window is below 5 minutes. This may cause adverse effects with routing.
  warnings.warn(
C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\.venv\lib\site-packages\r5py\r5\regional_task.py:266: RuntimeWarning: The provided departure time window is below 5 minutes. This may cause adverse effects with routing.
  warnings.warn(


Computing travel times for 750 origins x 750 destinations at 2026-01-09 07:30:00...
Computing travel times for 750 origins x 750 destinations at 2026-01-09 08:00:00...
Computing travel times for 750 origins x 750 destinations at 2026-01-09 08:30:00...
Aggregated multi-departure TTM rows: 562500
After conversion: 562500 rows, non-null travel_time: 189219
  QC constant check: top1_share=0.013, threshold=0.98, top_value=158.0
  QC sparse check: nonnull_share=0.336, threshold=0.01; nonnull_cnt=189219, min_abs=500
QC checks: suspicious_constant=False, too_sparse=False
R5 TTM accepted: 562500 rows (nonnull=189219, share=0.3364)
  Transit sanity: median_travel_time=128.0 min, nonnull=189219
Fill plan: |O|=750, |D|=750, expected_pairs=562500, r5_rows=562500, r5_nonnull=189219
Fill stats: filled_nonnull=193313, filled_by_walk~=4094, fill_fraction~=0.021
Filled matrix size: 562500 rows
Finished dublin: engine_mode=r5py_plus_walk_fill

=== PARIS ===
Departures: [datetime.datetime(2026, 1, 27, 7, 

C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\.venv\lib\site-packages\r5py\r5\regional_task.py:266: RuntimeWarning: The provided departure time window is below 5 minutes. This may cause adverse effects with routing.
  warnings.warn(
C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\.venv\lib\site-packages\r5py\r5\regional_task.py:266: RuntimeWarning: The provided departure time window is below 5 minutes. This may cause adverse effects with routing.
  warnings.warn(


Computing travel times for 750 origins x 750 destinations at 2026-01-27 07:30:00...
Computing travel times for 750 origins x 750 destinations at 2026-01-27 08:00:00...
Computing travel times for 750 origins x 750 destinations at 2026-01-27 08:30:00...
Aggregated multi-departure TTM rows: 562500
After conversion: 562500 rows, non-null travel_time: 274413
  QC constant check: top1_share=0.023, threshold=0.98, top_value=76.0
  QC sparse check: nonnull_share=0.488, threshold=0.01; nonnull_cnt=274413, min_abs=500
QC checks: suspicious_constant=False, too_sparse=False
R5 TTM accepted: 562500 rows (nonnull=274413, share=0.4878)
  Transit sanity: median_travel_time=74.0 min, nonnull=274413
Fill plan: |O|=750, |D|=750, expected_pairs=562500, r5_rows=562500, r5_nonnull=274413
Fill stats: filled_nonnull=293825, filled_by_walk~=19412, fill_fraction~=0.066
Filled matrix size: 562500 rows
Finished paris: engine_mode=r5py_plus_walk_fill

=== WARSAW ===
Using primary GTFS: C:\Users\Michc\Dropbox\IO_UW

...

C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\.venv\lib\site-packages\r5py\r5\regional_task.py:227: RuntimeWarning: Departure time 2025-11-13 08:00:00 is outside of the time range covered by currently loaded GTFS data sets.
  warnings.warn(


Computing travel times for 750 origins x 750 destinations at 2025-11-13 08:30:00...


C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\.venv\lib\site-packages\r5py\r5\regional_task.py:227: RuntimeWarning: Departure time 2025-11-13 08:30:00 is outside of the time range covered by currently loaded GTFS data sets.
  warnings.warn(


Aggregated multi-departure TTM rows: 562500
After conversion: 562500 rows, non-null travel_time: 365421
  QC constant check: top1_share=0.013, threshold=0.98, top_value=140.0
  QC sparse check: nonnull_share=0.650, threshold=0.01; nonnull_cnt=365421, min_abs=500
QC checks: suspicious_constant=False, too_sparse=False
R5 TTM accepted: 562500 rows (nonnull=365421, share=0.6496)
  Transit sanity: median_travel_time=121.0 min, nonnull=365421
Fill plan: |O|=750, |D|=750, expected_pairs=562500, r5_rows=562500, r5_nonnull=365421
Fill stats: filled_nonnull=367361, filled_by_walk~=1940, fill_fraction~=0.005
Filled matrix size: 562500 rows
Finished warsaw: engine_mode=r5py_plus_walk_fill


,city,engine_mode,n_rows,travel_time_min_nonnull_share,travel_time_min_min,travel_time_min_median,travel_time_min_p90,travel_time_min_max,share_capped_or_max_time
0,dublin,r5py_plus_walk_fill,562500,0.343668,0.0,127.0,168.0,179.0,0.0
1,paris,r5py_plus_walk_fill,562500,0.522356,0.0,73.0,95.0,138.0,0.0
2,warsaw,r5py_plus_walk_fill,562500,0.653086,0.0,121.0,164.0,179.0,0.0


## 9) Transformacja TTM → WTC (Whole Travel Chain Cost)

R5 (Conveyal) oblicza czas podróży jako **pełny Whole Travel Chain**:
- **Dojście (first mile)**: czas marszu od origin do najbliższego przystanku (sieć OSM)
- **Oczekiwanie**: R5 automatycznie uwzględnia headway-based waiting time
- **Przejazd (in-vehicle time)**: czas w pojeździe na wyznaczonej trasie
- **Przesiadki**: R5 uwzględnia czas przesiadek (walk + wait) w optymalnej trasie
- **Odejście (last mile)**: czas marszu od ostatniego przystanku do destination (sieć OSM)

Dlatego `travel_time_min` z R5 **jest już uogólnionym kosztem podróży (GTC/WTC)**.
Transformacja w tej komórce dokumentuje ten fakt i opcjonalnie pozwala na dodanie
dodatkowych wariantów kosztu (np. z karami za przesiadki, jeśli zostaną zidentyfikowane
w danych).

In [14]:
# --- Dokumentacja impedancji i WTC ---
# R5 travel_time_min jest PEŁNYM Whole Travel Chain (WTC):
#   c_ij = t_walk_first_mile + t_wait + t_in_vehicle + t_transfer + t_walk_last_mile
# Silnik routingowy R5 (Conveyal) optymalizuje trasę uwzględniając wszystkie
# składowe łańcucha podróży, headway-based waiting, przesiadki i marsz po OSM.
# Dlatego transformacja TTM → WTC jest dokumentacyjna (identity), a nie obliczeniowa.

IMPEDANCE_PARAMS = {
    'engine': 'r5py (Conveyal R5)',
    'wtc_components_included': [
        'first_mile_walk (OSM pedestrian network)',
        'waiting_time (headway-based, optimized by R5)',
        'in_vehicle_time (GTFS schedule)',
        'transfer_time (walk + wait at transfer stops)',
        'last_mile_walk (OSM pedestrian network)',
    ],
    'departure_time_window': list(DEPARTURE_TIME_WINDOW),
    'multi_departure_agg': MULTI_DEPARTURE_AGG,
    'walk_speed_kmh': WALK_SPEED_KMH,
    'max_walk_distance_m': MAX_WALK_DISTANCE_M,
    'max_trip_duration_min': MAX_TRIP_DURATION_MIN,
    'cost_variants': {
        'wtc_r5': {
            'definition': 'cost_min = travel_time_min (R5 full WTC)',
            'components': 'first_mile + wait + in_vehicle + transfer + last_mile',
            'note': 'R5 travel_time already includes all WTC components; no additional penalties applied',
        },
    },
}

for city in CITIES:
    city_dir = OUT_DIR / city
    ttm_path = city_dir / 'ttm' / 'ttm_raw.parquet'
    ttm = pd.read_parquet(ttm_path)

    wtc = ttm[['origin_id', 'dest_id', 'travel_time_min']].copy()
    # RC5: propaguj flagę walk-fill
    if 'is_walk_fill' in ttm.columns:
        wtc['is_walk_fill'] = ttm['is_walk_fill'].astype(bool)
    else:
        wtc['is_walk_fill'] = False
    wtc['cost_variant'] = 'wtc_r5'
    wtc.rename(columns={'travel_time_min': 'cost_min'}, inplace=True)
    # cost_min_strict: tylko wartości R5 (walk-only pary → NaN)
    wtc['cost_min_strict'] = wtc['cost_min'].where(~wtc['is_walk_fill'])

    (city_dir / 'wtc').mkdir(parents=True, exist_ok=True)
    (city_dir / 'wtc' / 'impedance_params.json').write_text(json.dumps(IMPEDANCE_PARAMS, ensure_ascii=False, indent=2), encoding='utf-8')
    wtc.to_parquet(city_dir / 'wtc' / 'wtc_matrix.parquet', index=False)

print('WTC matrices generated for all cities')
print(f'Cost variant: wtc_r5 (R5 full Whole Travel Chain)')
print(json.dumps(IMPEDANCE_PARAMS['wtc_components_included'], indent=2))

WTC matrices generated for all cities
Cost variant: wtc_r5 (R5 full Whole Travel Chain)
[
  "first_mile_walk (OSM pedestrian network)",
  "waiting_time (headway-based, optimized by R5)",
  "in_vehicle_time (GTFS schedule)",
  "transfer_time (walk + wait at transfer stops)",
  "last_mile_walk (OSM pedestrian network)"
]


## 10) Raporty per miasto i zbiorczy summary

Generujemy markdown summary dla każdego miasta oraz zbiorczy plik `summary.md`.


In [22]:
# Per-miasto reports/etap3_summary.md
for city in CITIES:
    city_dir = OUT_DIR / city

    od_val = json.loads((city_dir / 'qc' / 'validation_report.json').read_text(encoding='utf-8'))
    ttm_qc = pd.read_csv(city_dir / 'ttm' / 'ttm_qc_summary.csv').iloc[0].to_dict()

    # Run info (for departure datetimes)
    run_info_path = city_dir / 'run_info.json'
    city_run = json.loads(run_info_path.read_text(encoding='utf-8')) if run_info_path.exists() else {}

    lines = []
    lines.append(f'# Etap III — raport: {city}')
    lines.append('')
    lines.append(f"Uruchomiono (UTC): `{utc_now_iso()}`")
    lines.append('')

    # Parametry analizy
    lines.append('## Parametry')
    lines.append(f"- od_scale_mode: `{OD_SCALE_MODE}`")
    lines.append(f"- sample_k: `{OD_SAMPLE_K}`")
    lines.append(f"- origin_role: `{od_val['origin_role']}`")
    lines.append(f"- dest_role: `{od_val['dest_role']}`")
    lines.append(f"- engine_mode: `{ttm_qc['engine_mode']}`")
    lines.append(f"- admin_boundary_filter: `{od_val.get('admin_boundary_filter', 'N/A')}`")
    lines.append(f"- stops_proximity_buffer_m: `{od_val.get('stops_proximity_buffer_m', 'N/A')}`")
    lines.append(f"- walk_detour_factor: `{WALK_DETOUR_FACTOR}`")
    lines.append(f"- walk_cap_min: `{FALLBACK_WALK_ONLY_MAX_TIME_MIN}`")
    lines.append('')

    # Departure datetimes (reproducibility)
    dep_dts = city_run.get('departure_datetimes')
    if dep_dts:
        lines.append('## Departure datetimes (reproducibility)')
        for dt_str in dep_dts:
            lines.append(f"- `{dt_str}`")
        lines.append('')

    # Filtracja OD — odczyt z osobnych plików QC (od_filter_origins.json / od_filter_destinations.json)
    filter_o_path = city_dir / 'qc' / 'od_filter_origins.json'
    filter_d_path = city_dir / 'qc' / 'od_filter_destinations.json'
    if filter_o_path.exists() or filter_d_path.exists():
        lines.append('## Filtracja OD (granica administracyjna + bliskość przystanków)')
        for role_label, fpath in [('origins', filter_o_path), ('destinations', filter_d_path)]:
            if fpath.exists():
                rqc = json.loads(fpath.read_text(encoding='utf-8'))
                lines.append(f"### {role_label}")
                lines.append(f"- przed filtrem: {rqc.get('n_input', '?')}")
                lines.append(f"- po granicy admin: {rqc.get('n_after_admin_boundary', '?')}")
                lines.append(f"- po bliskości przystanków: {rqc.get('n_after_stop_proximity', '?')}")
                lines.append(f"- zachowane: {rqc.get('kept_share', '?'):.1%}" if isinstance(rqc.get('kept_share'), (int, float)) else f"- zachowane: {rqc.get('kept_share', '?')}")
        lines.append('')

    # Skala OD
    lines.append('## Skala OD')
    lines.append(f"- n_origins: {od_val['n_origins']}")
    lines.append(f"- n_destinations: {od_val['n_destinations']}")
    lines.append(f"- n_pairs: {od_val['n_pairs']}")
    lines.append('')

    # TTM results
    lines.append('## Wyniki TTM (travel_time_min)')
    lines.append(f"- non-null share: {ttm_qc['travel_time_min_nonnull_share']:.3f}")
    lines.append(f"- min / median / p90 / max (min): {ttm_qc['travel_time_min_min']:.1f} / {ttm_qc['travel_time_min_median']:.1f} / {ttm_qc['travel_time_min_p90']:.1f} / {ttm_qc['travel_time_min_max']:.1f}")
    lines.append(f"- share capped/max_time: {ttm_qc['share_capped_or_max_time']:.3f}")
    lines.append('')

    # WTC info
    lines.append('## WTC (Whole Travel Chain)')
    lines.append('- wariant kosztu: `wtc_r5` (R5 full WTC)')
    lines.append('- składowe: first_mile_walk + wait + in_vehicle + transfer + last_mile_walk')
    lines.append('- transformacja: identity (R5 travel_time = WTC)')
    lines.append('')

    # Warnings
    warn_path = city_dir / 'qc' / 'warnings.md'
    if warn_path.exists():
        lines.append('## QC — ostrzeżenia')
        lines.append(warn_path.read_text(encoding='utf-8').strip())
        lines.append('')

    # RC4: Dokumentacja temporalna
    _season_map = {
        'dublin': ('zima',   'styczeń 2026', 'GTFS_Realtime.zip, min=2026-01-09'),
        'paris':  ('zima',   'styczeń 2026',     'gouv_paris_gtfs-export.zip, min=2026-01-27'),
        'warsaw': ('jesień', 'listopad 2025', 'warsaw_merged.zip, min=2025-11-13'),
    }
    _sc = _season_map.get(city, ('?', '?', '?'))
    lines.append('## Kontekst temporalny (RC4)')
    lines.append(f'- Pora roku GTFS: **{_sc[0]}** ({_sc[1]})')
    lines.append(f'- Źródło GTFS: {_sc[2]}')
    lines.append(
        '⚠ Daty GTFS różnią się między miastami (Dublin lato, Paryż zima, Warszawa jesień) '
        '— porównania wymagają uwzględnienia sezonowości oferty.'
    )
    lines.append('')

    (city_dir / 'reports').mkdir(parents=True, exist_ok=True)
    (city_dir / 'reports' / 'etap3_summary.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')

# Zbiorcze summary.md
summary_lines = []
summary_lines.append('# Etap III — podsumowanie')
summary_lines.append('')
summary_lines.append(f"Uruchomiono (UTC): `{utc_now_iso()}`")
summary_lines.append('')
summary_lines.append('## Parametry')
summary_lines.append(f"- od_scale_mode: `{OD_SCALE_MODE}`")
summary_lines.append(f"- sample_k: `{OD_SAMPLE_K}`")
summary_lines.append(f"- departure_time_window: `{DEPARTURE_TIME_WINDOW}`")
summary_lines.append(f"- multi_departure_agg: `{MULTI_DEPARTURE_AGG}`")
summary_lines.append(f"- admin_boundary_filter: `{USE_ADMIN_BOUNDARY_FILTER}`")
summary_lines.append(f"- stops_proximity_buffer_m: `{STOPS_PROXIMITY_BUFFER_M}`")
summary_lines.append(f"- walk_detour_factor: `{WALK_DETOUR_FACTOR}`")
summary_lines.append(f"- walk_cap_min: `{FALLBACK_WALK_ONLY_MAX_TIME_MIN}`")
summary_lines.append(f"- paris_od_role_mode: `{PARIS_OD_ROLE_MODE}`")
summary_lines.append(f"- cost_variant: `wtc_r5` (R5 full WTC)")
summary_lines.append('')

summary_lines.append('## QC (TTM) — tabela zbiorcza')
summary_lines.append('')

# tabelka markdown z TTM_QC
cols = ['city', 'engine_mode', 'n_rows', 'travel_time_min_nonnull_share',
        'travel_time_min_median', 'travel_time_min_p90', 'travel_time_min_max',
        'share_capped_or_max_time']
md_df = TTM_QC[cols].copy()
summary_lines.append('| ' + ' | '.join(cols) + ' |')
summary_lines.append('| ' + ' | '.join(['---'] * len(cols)) + ' |')
for _, r in md_df.iterrows():
    summary_lines.append('| ' + ' | '.join(str(r[c]) for c in cols) + ' |')
summary_lines.append('')

# QC porównawcze między miastami
summary_lines.append('## QC porównawcze (cross-city)')
summary_lines.append('')

# Sprawdzenie porównywalności parametrów
summary_lines.append('### Porównywalność parametrów')
summary_lines.append(f"- Wspólna siatka odniesienia: grid 1 km × 1 km")
summary_lines.append(f"- Wspólne okno departures: `{DEPARTURE_TIME_WINDOW}`")
summary_lines.append(f"- Wspólne parametry marszu: walk_speed={WALK_SPEED_KMH:.1f} km/h, max_walk={MAX_WALK_DISTANCE_M} m")
summary_lines.append(f"- Wspólny max_trip_duration: {MAX_TRIP_DURATION_MIN} min")
summary_lines.append(f"- Paris OD: `{PARIS_OD_ROLE_MODE}` (grid_to_grid = porównywalne z Dublin/Warsaw)")
summary_lines.append('')

# Ostrzeżenia ogólne
engine_modes = TTM_QC['engine_mode'].unique().tolist()
if len(set(engine_modes)) > 1:
    summary_lines.append('### ⚠ Ostrzeżenia porównawcze')
    summary_lines.append(f"- Różne engine_modes między miastami: {engine_modes}")
    summary_lines.append('- Porównanie wyników między miastami z różnymi engine_modes wymaga ostrożności')
    summary_lines.append('- Wyniki walk_only_fallback są przybliżeniem (euclidean * detour_factor), nie pełnym routingiem')
    summary_lines.append('')

# Median travel time comparison
if len(TTM_QC) >= 2:
    med_col = 'travel_time_min_median'
    medians = TTM_QC.set_index('city')[med_col]
    summary_lines.append('### Porównanie median czasu podróży')
    for c in medians.index:
        summary_lines.append(f"- {c}: {medians[c]:.1f} min")
    if medians.max() > 0:
        ratio = medians.max() / max(medians.min(), 0.1)
        summary_lines.append(f"- Stosunek max/min mediany: {ratio:.1f}x")
        if ratio > 3.0:
            summary_lines.append(f"- ⚠ Duża dysproporcja median (>{3.0}x) — zweryfikować engine_mode i pokrycie OD")
    summary_lines.append('')

# RC4: Ograniczenia temporalne
summary_lines.append('## Ograniczenia temporalne (RC4)')
summary_lines.append('')
summary_lines.append('| Miasto | GTFS mid-date | Pora roku | Uwaga |')
summary_lines.append('| --- | --- | --- | --- |')
summary_lines.append('| Dublin  | 2026-01-09 | **zima**    | Możliwe ograniczenia sezonowe |')
summary_lines.append('| Paryż   | 2026-01-27 | **zima**    | Możliwe ograniczenia sezonowe |')
summary_lines.append('| Warszawa | 2025-11-13 | **jesień** | Rozkład ZTM jesienny |')
summary_lines.append('')
summary_lines.append(
    '**Ograniczenie**: sezonowe różnice w oferowaniu usług mogą wpływać na porównywałność '
    'wskaźników dostępności między miastami (wymaga uwzględnienia w Etapie V).'
)
summary_lines.append('')

(OUT_DIR / 'summary.md').write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')

print('Summary reports generated')
print('Per-city reports:', [f'{c}/reports/etap3_summary.md' for c in CITIES])
print('Global summary:', OUT_DIR / 'summary.md')

Summary reports generated
Per-city reports: ['dublin/reports/etap3_summary.md', 'paris/reports/etap3_summary.md', 'warsaw/reports/etap3_summary.md']
Global summary: C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\outputs\etap3\summary.md


## 11) Artifacts index (kontrakt wyjściowy)

Zapisujemy indeks wszystkich wygenerowanych artefaktów do `artifacts_index.json` (analogicznie do Etapu I i II).


In [23]:
# artifacts_index.json Etapu III
artifacts_etap3 = {
    'generated_at_utc': utc_now_iso(),
    'od': {},
    'ttm': {},
    'wtc': {},
    'qc': {},
    'reports': {},
}

for city in CITIES:
    city_dir = OUT_DIR / city
    artifacts_etap3['od'][city] = {
        'origins_geojson': _rel(city_dir / 'od' / 'origins.geojson'),
        'destinations_geojson': _rel(city_dir / 'od' / 'destinations.geojson'),
        'od_units_csv': _rel(city_dir / 'od' / 'od_units.csv'),
    }
    artifacts_etap3['ttm'][city] = {
        'ttm_raw_parquet': _rel(city_dir / 'ttm' / 'ttm_raw.parquet'),
        'ttm_qc_summary_csv': _rel(city_dir / 'ttm' / 'ttm_qc_summary.csv'),
    }
    artifacts_etap3['wtc'][city] = {
        'impedance_params_json': _rel(city_dir / 'wtc' / 'impedance_params.json'),
        'wtc_matrix_parquet': _rel(city_dir / 'wtc' / 'wtc_matrix.parquet'),
    }
    artifacts_etap3['qc'][city] = {
        'validation_report_json': _rel(city_dir / 'qc' / 'validation_report.json'),
    }

    # Add warnings only if file exists
    if (city_dir / 'qc' / 'warnings.md').exists():
        artifacts_etap3['qc'][city]['warnings_md'] = _rel(city_dir / 'qc' / 'warnings.md')

    artifacts_etap3['reports'][city] = {
        'etap3_summary_md': _rel(city_dir / 'reports' / 'etap3_summary.md'),
    }

(OUT_DIR / 'artifacts_index.json').write_text(json.dumps(artifacts_etap3, ensure_ascii=False, indent=2), encoding='utf-8')

print('Wrote:', OUT_DIR / 'summary.md')
print('Wrote:', OUT_DIR / 'artifacts_index.json')


Wrote: C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\outputs\etap3\summary.md
Wrote: C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\outputs\etap3\artifacts_index.json


## 12) Podsumowanie i następne kroki

Etap III został ukończony. Wygenerowane artefakty:

- **OD units**: origins/destinations per miasto (GeoJSON + CSV z atrybutami)
- **TTM**: surowe macierze czasów podróży (parquet)
- **WTC**: macierze kosztów uogólnionych (parquet)
- **QC**: walidacje i ostrzeżenia per miasto
- **Reports**: markdown summaries per miasto + zbiorcze

**Następne kroki (Etap IV/V):**
- Implementacja modeli grawitacyjnych (gravity models)
- Analiza nierówności przestrzennych (spatial equity)
- Wizualizacje i mapy dostępności
- Komparatywna analiza między miastami


In [24]:
# Podgląd summary
print((OUT_DIR / 'summary.md').read_text(encoding='utf-8'))


# Etap III — podsumowanie

Uruchomiono (UTC): `2026-03-23T22:55:25Z`

## Parametry
- od_scale_mode: `sample_k`
- sample_k: `750`
- departure_time_window: `('07:30:00', '09:00:00')`
- multi_departure_agg: `min`
- admin_boundary_filter: `True`
- stops_proximity_buffer_m: `2000`
- walk_detour_factor: `1.3`
- walk_cap_min: `90.0`
- paris_od_role_mode: `grid_to_grid`
- cost_variant: `wtc_r5` (R5 full WTC)

## QC (TTM) — tabela zbiorcza

| city | engine_mode | n_rows | travel_time_min_nonnull_share | travel_time_min_median | travel_time_min_p90 | travel_time_min_max | share_capped_or_max_time |
| --- | --- | --- | --- | --- | --- | --- | --- |
| dublin | r5py_plus_walk_fill | 562500 | 0.34366755555555556 | 127.0 | 168.0 | 179.0 | 0.0 |
| paris | r5py_plus_walk_fill | 562500 | 0.5223555555555556 | 73.0 | 95.0 | 138.0 | 0.0 |
| warsaw | r5py_plus_walk_fill | 562500 | 0.6530862222222222 | 121.0 | 164.0 | 179.0 | 0.0 |

## QC porównawcze (cross-city)

### Porównywalność parametrów
- Wspólna siat